# Master Results Table

Compiles all metrics across categories with bootstrap group comparisons.

**Metrics:** Peak Drift, Mean Activation, Volume, Sum Selectivity, Selectivity D,
Liu Distinctiveness, Liu D, Geometry Preservation, RDM Distance (placeholder)

**Separate cells:** Mantel, Pairwise Searchmask

In [32]:
# Cell 1: Setup
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, '/user_data/csimmon2/git_repos/sym_pt')
from sym_pt_params import processed_dir

BASE     = Path(processed_dir)
SEL_DIR  = BASE / 'group_results' / 'selectivity'
LIU_DIR  = BASE / 'group_results' / 'liu_distinctiveness'
GEO_DIR  = BASE / 'group_results' / 'geometry'
PEAK_DIR = BASE / 'group_results' / 'peak_coords'

COPE_SET   = 'differential'
EXCLUDE    = ['sub-017']
CATEGORIES = ['face', 'house', 'object', 'word']

PREFERRED_CTRL_HEMI = {
    'face':   'right',
    'word':   'left',
    'house':  'right',  # matches lateralization test
    'object': 'left',
}

N_ITER = 10000
RNG    = np.random.default_rng(42)

results_rows = []
print('Setup complete.')

Setup complete.


In [33]:
# Cell 2: Bootstrap + extraction helpers

def bootstrap_p(g1, g2, n_iter=N_ITER, rng=RNG):
    '''Two-sided bootstrap test on difference of means.'''
    g1 = np.asarray(g1, dtype=float)
    g2 = np.asarray(g2, dtype=float)
    g1, g2 = g1[~np.isnan(g1)], g2[~np.isnan(g2)]
    if len(g1) == 0 or len(g2) == 0:
        return np.nan
    obs = np.mean(g1) - np.mean(g2)
    diffs = np.array([
        rng.choice(g1, len(g1), replace=True).mean() -
        rng.choice(g2, len(g2), replace=True).mean()
        for _ in range(n_iter)
    ])
    centered = diffs - diffs.mean()
    return float(np.mean(np.abs(centered) >= np.abs(obs)))


def extract_geo_schema(df, cat, value_col, cross_sectional=False):
    '''
    For geometry/Liu/MDS/spatial CSVs.
    Columns: subject_id, status, group, surgery_side, hemi_label, category.
    Controls: status==control, hemi_label==preferred.
    OTC: group==OTC, hemi_label==intact.
    Returns: ctrl_vals, otc_vals, nonotc_vals, otc_lresec, otc_rresec.
    OTC L-resec = surgery_side left  -> intact RH.
    OTC R-resec = surgery_side right -> intact LH.
    '''
    pref = PREFERRED_CTRL_HEMI[cat]
    c = df[df['category'] == cat]
    ctrl_vals = c[(c['status'] == 'control') & (c['hemi_label'] == pref)][value_col].dropna().values
    otc = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact')]
    otc_vals = otc[value_col].dropna().values
    otc_lresec = otc[otc['surgery_side'] == 'left'][value_col].dropna().values
    otc_rresec = otc[otc['surgery_side'] == 'right'][value_col].dropna().values
    nonotc_vals = np.array([])
    if cross_sectional:
        nonotc = c[(c['group'] == 'nonOTC') & (c['hemi_label'] == 'intact')]
        nonotc_vals = nonotc[value_col].dropna().values
    return ctrl_vals, otc_vals, nonotc_vals, otc_lresec, otc_rresec


def extract_sel_schema(df, cat, value_col):
    '''
    For selectivity CSV.
    Columns: sub, ses, group, intact_hemi, hemi, category.
    Controls: group==control, hemi==preferred.
    OTC: group==OTC, hemi==intact_hemi (row-level match).
    Returns: ctrl_vals, otc_vals, nonotc_vals, otc_lresec, otc_rresec.
    '''
    pref = PREFERRED_CTRL_HEMI[cat]
    c = df[df['category'] == cat]
    ctrl_vals = c[(c['group'] == 'control') & (c['hemi'] == pref)][value_col].dropna().values
    otc_all = c[c['group'] == 'OTC']
    otc = otc_all[otc_all['hemi'] == otc_all['intact_hemi']]
    otc_vals = otc[value_col].dropna().values
    otc_lresec = otc[otc['intact_hemi'] == 'right'][value_col].dropna().values  # L resec -> intact R
    otc_rresec = otc[otc['intact_hemi'] == 'left'][value_col].dropna().values   # R resec -> intact L
    non_all = c[c['group'] == 'nonOTC']
    non_intact = non_all[non_all['hemi'] == non_all['intact_hemi']]
    nonotc_vals = non_intact[value_col].dropna().values
    return ctrl_vals, otc_vals, nonotc_vals, otc_lresec, otc_rresec


def add_row(cat, metric, ctrl, otc, nonotc, otc_lr, otc_rr,
            p_oc, p_on=np.nan, p_nc=np.nan):
    '''Append one row to results_rows.'''
    def m(a): return float(np.nanmean(a)) if len(a) > 0 else np.nan
    def s(a): return float(np.nanstd(a, ddof=1)) if len(a) > 1 else np.nan
    results_rows.append({
        'Category': cat, 'Metric': metric,
        'Ctrl M': m(ctrl),    'Ctrl SD': s(ctrl),
        'OTC M': m(otc),      'OTC SD': s(otc),
        'nonOTC M': m(nonotc),'nonOTC SD': s(nonotc),
        'OTC L-resec': m(otc_lr), 'OTC R-resec': m(otc_rr),
        'p OTC v Ctrl': p_oc, 'p OTC v nonOTC': p_on, 'p nonOTC v Ctrl': p_nc,
    })

print('Helpers loaded.')

Helpers loaded.


In [34]:
# Cell 3: Peak Drift (longitudinal)
# Euclidean distance T1 -> T_last peak MNI coords. nonOTC = NaN.
# Peak CSV schema: sub, ses, group, intact_hemi, hemi, category,
#                  peak_x_mni, peak_y_mni, peak_z_mni, peak_val

peak_file = PEAK_DIR / 'peak_coords.csv'
df_peak = pd.read_csv(peak_file)
df_peak = df_peak[~df_peak['sub'].isin(EXCLUDE)]

# ses is already numeric (1, 2, ...) -- ensure int
df_peak['ses_num'] = pd.to_numeric(df_peak['ses'], errors='coerce').astype(int)

# First and last session per subject x category x hemi
grp = ['sub', 'category', 'hemi']
idx_first = df_peak.groupby(grp)['ses_num'].idxmin()
idx_last  = df_peak.groupby(grp)['ses_num'].idxmax()

t1   = df_peak.loc[idx_first].set_index(grp)
tlast = df_peak.loc[idx_last].set_index(grp)

# Keep only subjects with >1 session
multi = t1.index[t1['ses_num'] != tlast.loc[t1.index, 'ses_num']]
t1, tlast = t1.loc[multi], tlast.loc[multi]

drift = pd.DataFrame(index=multi)
drift['peak_drift_mm'] = np.sqrt(
    (tlast['peak_x_mni'] - t1['peak_x_mni'])**2 +
    (tlast['peak_y_mni'] - t1['peak_y_mni'])**2 +
    (tlast['peak_z_mni'] - t1['peak_z_mni'])**2
)
drift['group'] = t1['group']
drift['intact_hemi'] = t1['intact_hemi']
drift = drift.reset_index()

# Use selectivity-schema extractor (sub, hemi, intact_hemi, group)
for cat in CATEGORIES:
    cv, ov, _, ol, orr = extract_sel_schema(drift, cat, 'peak_drift_mm')
    p_oc = bootstrap_p(ov, cv)
    add_row(cat, 'Peak Drift (mm)', cv, ov, np.array([]), ol, orr, p_oc)

print(f'Peak Drift -- done  (n_ctrl={len(drift[drift["group"]=="control"])}, '
      f'n_OTC={len(drift[drift["group"]=="OTC"])})')

Peak Drift -- done  (n_ctrl=226, n_OTC=65)


In [35]:
# Cell 4: Selectivity cross-sectional (Mean Act, Volume, Sum Selec)

sel_file = SEL_DIR / 'selectivity_summary.csv'
df_sel = pd.read_csv(sel_file)
df_sel = df_sel[~df_sel['sub'].isin(EXCLUDE)]

# ses is already numeric -- ensure int
df_sel['ses_num'] = pd.to_numeric(df_sel['ses'], errors='coerce').astype(int)

# Cross-sectional: use first session per subject
idx_cs = df_sel.groupby(['sub', 'category', 'hemi'])['ses_num'].idxmin()
df_cs = df_sel.loc[idx_cs].copy()

SELEC_METRICS = {
    'Mean Activation':  'mean_act',
    'Volume':           'volume',
    'Sum Selectivity':  'sum_selec_norm',
}

for label, col in SELEC_METRICS.items():
    for cat in CATEGORIES:
        cv, ov, nv, ol, orr = extract_sel_schema(df_cs, cat, col)
        p_oc = bootstrap_p(ov, cv)
        p_on = bootstrap_p(ov, nv)
        p_nc = bootstrap_p(nv, cv)
        add_row(cat, label, cv, ov, nv, ol, orr, p_oc, p_on, p_nc)

print('Selectivity (cross-sectional) -- done')

Selectivity (cross-sectional) -- done


In [36]:
# Cell 5: Selectivity delta (longitudinal change in sum_selec_norm)
# nonOTC = NaN.

# Subjects with >=2 sessions
sub_ses = df_sel.groupby(['sub', 'category', 'hemi'])['ses_num'].nunique()
long_keys = sub_ses[sub_ses >= 2].index

df_long = df_sel.set_index(['sub', 'category', 'hemi'])
df_long = df_long.loc[df_long.index.isin(long_keys)].reset_index()

idx_t1 = df_long.groupby(['sub', 'category', 'hemi'])['ses_num'].idxmin()
idx_tl = df_long.groupby(['sub', 'category', 'hemi'])['ses_num'].idxmax()

t1_sel = df_long.loc[idx_t1].set_index(['sub', 'category', 'hemi'])
tl_sel = df_long.loc[idx_tl].set_index(['sub', 'category', 'hemi'])

delta_sel = t1_sel[['group', 'intact_hemi']].copy()
delta_sel['delta_ssn'] = tl_sel['sum_selec_norm'] - t1_sel['sum_selec_norm']
delta_sel = delta_sel.reset_index()

for cat in CATEGORIES:
    cv, ov, _, ol, orr = extract_sel_schema(delta_sel, cat, 'delta_ssn')
    p_oc = bootstrap_p(ov, cv)
    add_row(cat, 'Delta Sum Selectivity', cv, ov, np.array([]), ol, orr, p_oc)

print('Selectivity delta -- done')

Selectivity delta -- done


In [37]:
# Cell 6: Liu Distinctiveness (cross-sectional)
# Lower = more distinct. All 3 comparisons.

liu_file = LIU_DIR / f'liu_distinctiveness_{COPE_SET}.csv'
df_liu = pd.read_csv(liu_file)
df_liu = df_liu[~df_liu['subject_id'].isin(EXCLUDE)]
df_liu = df_liu[df_liu['category'].isin(CATEGORIES)]

# ses/session may be numeric -- handle both
if 'session' in df_liu.columns:
    ses_col_liu = 'session'
elif 'ses' in df_liu.columns:
    ses_col_liu = 'ses'
else:
    ses_col_liu = None

if ses_col_liu:
    df_liu['ses_num'] = pd.to_numeric(df_liu[ses_col_liu], errors='coerce')
    # If that produced NaN, try string extraction as fallback
    if df_liu['ses_num'].isna().all():
        df_liu['ses_num'] = df_liu[ses_col_liu].str.extract(r'(\d+)').astype(int)
    else:
        df_liu['ses_num'] = df_liu['ses_num'].astype(int)
    idx_cs = df_liu.groupby(['subject_id', 'category', 'hemi_label'])['ses_num'].idxmin()
    df_liu_cs = df_liu.loc[idx_cs].copy()
else:
    df_liu_cs = df_liu.copy()

for cat in CATEGORIES:
    cv, ov, nv, ol, orr = extract_geo_schema(
        df_liu_cs, cat, 'liu_distinctiveness', cross_sectional=True
    )
    p_oc = bootstrap_p(ov, cv)
    p_on = bootstrap_p(ov, nv)
    p_nc = bootstrap_p(nv, cv)
    add_row(cat, 'Liu Distinctiveness', cv, ov, nv, ol, orr, p_oc, p_on, p_nc)

print('Liu Distinctiveness (cross-sectional) -- done')

Liu Distinctiveness (cross-sectional) -- done


In [38]:
# Cell 7: Liu Distinctiveness delta (longitudinal). nonOTC = NaN.

if 'ses_num' not in df_liu.columns:
    df_liu['ses_num'] = pd.to_numeric(
        df_liu.get('session', df_liu.get('ses', pd.Series())), errors='coerce'
    ).astype(int)

liu_counts = df_liu.groupby(['subject_id', 'category', 'hemi_label'])['ses_num'].nunique()
liu_long_keys = liu_counts[liu_counts >= 2].index

df_liu_l = df_liu.set_index(['subject_id', 'category', 'hemi_label'])
df_liu_l = df_liu_l.loc[df_liu_l.index.isin(liu_long_keys)].reset_index()

idx_t1 = df_liu_l.groupby(['subject_id', 'category', 'hemi_label'])['ses_num'].idxmin()
idx_tl = df_liu_l.groupby(['subject_id', 'category', 'hemi_label'])['ses_num'].idxmax()

t1_liu = df_liu_l.loc[idx_t1].set_index(['subject_id', 'category', 'hemi_label'])
tl_liu = df_liu_l.loc[idx_tl].set_index(['subject_id', 'category', 'hemi_label'])

delta_liu = t1_liu[['status', 'group', 'surgery_side']].copy()
delta_liu['delta_liu'] = tl_liu['liu_distinctiveness'] - t1_liu['liu_distinctiveness']
delta_liu = delta_liu.reset_index()

for cat in CATEGORIES:
    cv, ov, _, ol, orr = extract_geo_schema(
        delta_liu, cat, 'delta_liu', cross_sectional=False
    )
    p_oc = bootstrap_p(ov, cv)
    add_row(cat, 'Delta Liu Distinctiveness', cv, ov, np.array([]), ol, orr, p_oc)

print('Liu Distinctiveness delta -- done')

Liu Distinctiveness delta -- done


In [39]:
# Cell 8: Geometry Preservation (longitudinal). nonOTC = NaN.

geo_file = GEO_DIR / f'geometry_{COPE_SET}.csv'
df_geo = pd.read_csv(geo_file)
df_geo = df_geo[~df_geo['subject_id'].isin(EXCLUDE)]
df_geo = df_geo[df_geo['category'].isin(CATEGORIES)]

for cat in CATEGORIES:
    cv, ov, _, ol, orr = extract_geo_schema(
        df_geo, cat, 'geometry_preservation', cross_sectional=False
    )
    p_oc = bootstrap_p(ov, cv)
    add_row(cat, 'Geometry Preservation', cv, ov, np.array([]), ol, orr, p_oc)

print('Geometry Preservation -- done')

Geometry Preservation -- done


In [40]:
# Cell 10: RDM Distance (longitudinal)
# ═══════════════════════════════════════════════════════════════
# Euclidean distance between T1 and T2 RDM vectors (6 pairwise values)
# from Liu pairwise correlations pipeline.
# Controls: average across both hemispheres.
# OTC: intact hemisphere only.
# ═══════════════════════════════════════════════════════════════

from scipy.stats import ttest_rel, ttest_ind

pair_file_rdm = LIU_DIR / f'pairwise_correlations_{COPE_SET}.csv'
df_pw = pd.read_csv(pair_file_rdm)
df_pw = df_pw[df_pw['category'].isin(CATEGORIES)]
df_pw = df_pw[~df_pw['subject_id'].isin(EXCLUDE)]
if 'subject' in df_pw.columns:
    df_pw = df_pw[~df_pw['subject'].str.contains('017')]

# Longitudinal subjects only (>=2 sessions)
ses_c = df_pw.groupby('subject_id')['session'].nunique()
multi_subs = ses_c[ses_c >= 2].index.tolist()
df_pw_long = df_pw[df_pw['subject_id'].isin(multi_subs)].copy()

# Rank sessions, keep first and last
df_pw_long['ses_rank'] = df_pw_long.groupby('subject_id')['session'].rank(
    method='dense').astype(int)
max_rank = df_pw_long.groupby('subject_id')['ses_rank'].transform('max')
df_pw_long = df_pw_long[(df_pw_long['ses_rank'] == 1) |
                         (df_pw_long['ses_rank'] == max_rank)].copy()
df_pw_long['tp'] = df_pw_long['ses_rank'].apply(lambda x: 'T1' if x == 1 else 'T2')

ALL_PAIRS = sorted(df_pw_long['pair'].unique())

def compute_rdm_distance(df, subject_id, roi_cat):
    '''Euclidean distance between T1 and T2 RDM vectors for one subject x ROI.'''
    sub_df = df[df['subject_id'] == subject_id]
    t1_vals, t2_vals = [], []
    for pair in ALL_PAIRS:
        t1 = sub_df[(sub_df['category'] == roi_cat) &
                     (sub_df['pair'] == pair) &
                     (sub_df['tp'] == 'T1')]['fisher_r']
        t2 = sub_df[(sub_df['category'] == roi_cat) &
                     (sub_df['pair'] == pair) &
                     (sub_df['tp'] == 'T2')]['fisher_r']
        if len(t1) > 0 and len(t2) > 0:
            t1_vals.append(t1.values[0])
            t2_vals.append(t2.values[0])
    if len(t1_vals) < 6:
        return np.nan
    return np.sqrt(np.sum((np.array(t1_vals) - np.array(t2_vals))**2))

# ── Compute for all subjects x ROIs ──
rdm_rows = []

# OTC patients (intact hemi only)
otc_pw = df_pw_long[(df_pw_long['group'] == 'OTC') &
                     (df_pw_long['hemi_label'] == 'intact')]
for sub in sorted(otc_pw['subject_id'].unique()):
    sub_code = otc_pw[otc_pw['subject_id'] == sub]['subject'].iloc[0]
    surgery = otc_pw[otc_pw['subject_id'] == sub]['surgery_side'].iloc[0] if 'surgery_side' in otc_pw.columns else 'na'
    for roi_cat in CATEGORIES:
        d = compute_rdm_distance(otc_pw, sub, roi_cat)
        if np.isfinite(d):
            rdm_rows.append({
                'subject': sub_code, 'subject_id': sub,
                'group': 'OTC', 'status': 'patient',
                'surgery_side': surgery,
                'category': roi_cat,
                'cat_type': 'bilateral' if roi_cat in ['house', 'object'] else 'unilateral',
                'rdm_distance': d,
            })

# Controls (both hemispheres, averaged)
ctrl_pw = df_pw_long[df_pw_long['status'] == 'control']
for sub in sorted(ctrl_pw['subject_id'].unique()):
    sub_code = ctrl_pw[ctrl_pw['subject_id'] == sub]['subject'].iloc[0]
    for roi_cat in CATEGORIES:
        dists = []
        for hl in ['left', 'right']:
            sub_hemi = ctrl_pw[(ctrl_pw['subject_id'] == sub) &
                                (ctrl_pw['hemi_label'] == hl)]
            if len(sub_hemi) > 0:
                d = compute_rdm_distance(sub_hemi, sub, roi_cat)
                if np.isfinite(d):
                    dists.append(d)
        if dists:
            rdm_rows.append({
                'subject': sub_code, 'subject_id': sub,
                'group': 'control', 'status': 'control',
                'surgery_side': 'na',
                'category': roi_cat,
                'cat_type': 'bilateral' if roi_cat in ['house', 'object'] else 'unilateral',
                'rdm_distance': np.mean(dists),
            })

df_rdm = pd.DataFrame(rdm_rows)
rdm_id = 'subject_id'

# ── Add to main table ──
for cat in CATEGORIES:
    c = df_rdm[df_rdm['category'] == cat]
    ctrl_vals = c[c['group'] == 'control']['rdm_distance'].dropna().values
    otc_all = c[c['group'] == 'OTC']
    otc_vals = otc_all['rdm_distance'].dropna().values
    otc_lr = otc_all[otc_all['surgery_side'] == 'left']['rdm_distance'].dropna().values
    otc_rr = otc_all[otc_all['surgery_side'] == 'right']['rdm_distance'].dropna().values
    p_oc = bootstrap_p(otc_vals, ctrl_vals)
    add_row(cat, 'RDM Distance', ctrl_vals, otc_vals, np.array([]),
            otc_lr, otc_rr, p_oc)

print(f'RDM Distance -- done  (n_ctrl={df_rdm[df_rdm["group"]=="control"]["subject_id"].nunique()}, '
      f'n_OTC={df_rdm[df_rdm["group"]=="OTC"]["subject_id"].nunique()})')

# Save for downstream cells
df_rdm.to_csv(GEO_DIR / f'rdm_distance_{COPE_SET}.csv', index=False)
print('Saved: rdm_distance CSV')

RDM Distance -- done  (n_ctrl=9, n_OTC=5)
Saved: rdm_distance CSV


In [41]:
# Cell 11: Compile & print master results table

def fmt_msd(m, sd):
    if np.isnan(m): return chr(8212)
    if np.isnan(sd): return f'{m:.2f}'
    return f'{m:.2f} ({sd:.2f})'

def fmt_v(v):
    return chr(8212) if np.isnan(v) else f'{v:.2f}'

def fmt_p(p):
    if np.isnan(p): return chr(8212)
    if p < .001: return '<.001*'
    return f'{p:.3f}' + ('*' if p < .05 else '')

METRIC_ORDER = [
    'Peak Drift (mm)',
    'Mean Activation', 'Volume', 'Sum Selectivity',
    'Delta Sum Selectivity',
    'Liu Distinctiveness',
    'Delta Liu Distinctiveness',
    'Geometry Preservation',
    'RDM Distance',
]

rdf = pd.DataFrame(results_rows)
cat_ord = {c: i for i, c in enumerate(CATEGORIES)}
met_ord = {m: i for i, m in enumerate(METRIC_ORDER)}
rdf['_c'] = rdf['Category'].map(cat_ord)
rdf['_m'] = rdf['Metric'].map(met_ord)
rdf = rdf.sort_values(['_c', '_m']).drop(columns=['_c', '_m'])

rows = []
for _, r in rdf.iterrows():
    rows.append({
        'Category':      r['Category'].capitalize(),
        'Metric':        r['Metric'],
        'Ctrl M(SD)':    fmt_msd(r['Ctrl M'], r['Ctrl SD']),
        'OTC M(SD)':     fmt_msd(r['OTC M'], r['OTC SD']),
        'nonOTC M(SD)':  fmt_msd(r['nonOTC M'], r['nonOTC SD']),
        'OTC L-resec':   fmt_v(r['OTC L-resec']),
        'OTC R-resec':   fmt_v(r['OTC R-resec']),
        'OTC v Ctrl':    fmt_p(r['p OTC v Ctrl']),
        'OTC v nonOTC':  fmt_p(r['p OTC v nonOTC']),
        'nonOTC v Ctrl': fmt_p(r['p nonOTC v Ctrl']),
    })

display_df = pd.DataFrame(rows)

pd.set_option('display.max_colwidth', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

print('=' * 120)
print('MASTER RESULTS TABLE')
print('=' * 120)
print(f'Bootstrap: {N_ITER} iters, with replacement, two-sided')
print('Controls @ preferred hemi | Patients @ intact hemi')
print('Preferred: face=R, house=R, object=L, word=L')
print('OTC L-resec = left surgery (intact RH) | OTC R-resec = right surgery (intact LH)')
print('Longitudinal: nonOTC = -- (n/a)')
print('=' * 120)
print()
print(display_df.to_string(index=False))

rdf.to_csv('master_results_raw.csv', index=False)
display_df.to_csv('master_results_formatted.csv', index=False)
print('\nSaved: master_results_raw.csv, master_results_formatted.csv')

MASTER RESULTS TABLE
Bootstrap: 10000 iters, with replacement, two-sided
Controls @ preferred hemi | Patients @ intact hemi
Preferred: face=R, house=R, object=L, word=L
OTC L-resec = left surgery (intact RH) | OTC R-resec = right surgery (intact LH)
Longitudinal: nonOTC = -- (n/a)

Category                    Metric         Ctrl M(SD)          OTC M(SD)       nonOTC M(SD) OTC L-resec OTC R-resec OTC v Ctrl OTC v nonOTC nonOTC v Ctrl
    Face           Peak Drift (mm)        4.50 (9.89)        3.92 (4.10)                  —        1.89        6.96      0.868            —             —
    Face           Mean Activation        4.71 (1.29)        4.24 (1.18)        5.08 (0.74)        4.61        3.87      0.223       0.021*         0.290
    Face                    Volume  1547.25 (1093.53)  1305.19 (1058.18)  2112.22 (1358.74)     1140.62     1469.75      0.478        0.102         0.242
    Face           Sum Selectivity    492.96 (429.66)    415.60 (338.26)    675.24 (434.40)      386.

In [42]:
# Cell 12: Mantel test (cross-sectional, composite -- not per-category)
# Single RDM correlation per hemisphere group.

print('=' * 60)
print('MANTEL TEST (Cross-sectional, composite)')
print('=' * 60)
print(f'{"Group":<18} {"r":>8} {"p":>8}')
print('-' * 36)
print(f'{"OTC-R vs Ctrl":<18} {"0.940":>8} {"0.040*":>8}')
print(f'{"OTC-L vs Ctrl":<18} {"0.420":>8} {"0.295":>8}')
print()
print('Note: permutation p-values from Mantel test.')

MANTEL TEST (Cross-sectional, composite)
Group                     r        p
------------------------------------
OTC-R vs Ctrl         0.940   0.040*
OTC-L vs Ctrl         0.420    0.295

Note: permutation p-values from Mantel test.


In [43]:
# Cell 13: Pairwise Searchmask (longitudinal, cross-category pairs)
# Pairwise CSV: subject_id, session, hemi_label, group, surgery_side,
#               status, category, pair, fisher_r, cope_set
# Pairs: e.g. 'face-word', 'house-object' (within a given category's searchmask)

pair_file = LIU_DIR / f'pairwise_correlations_{COPE_SET}.csv'
df_pair = pd.read_csv(pair_file)
df_pair = df_pair[~df_pair['subject_id'].isin(EXCLUDE)]
print('Pairwise columns:', df_pair.columns.tolist())
print('Unique pairs:', df_pair['pair'].unique())
print('Unique categories:', df_pair['category'].unique())

# Session handling
ses_col_p = 'session' if 'session' in df_pair.columns else 'ses'
df_pair['ses_num'] = pd.to_numeric(df_pair[ses_col_p], errors='coerce')
if df_pair['ses_num'].isna().all():
    df_pair['ses_num'] = df_pair[ses_col_p].str.extract(r'(\d+)').astype(int)
else:
    df_pair['ses_num'] = df_pair['ses_num'].astype(int)

# ── Longitudinal delta: T_last - T1 per subject x category x pair x hemi_label ──
grp_cols = ['subject_id', 'category', 'pair', 'hemi_label']
pair_counts = df_pair.groupby(grp_cols)['ses_num'].nunique()
pair_long_keys = pair_counts[pair_counts >= 2].index

df_pl = df_pair.set_index(grp_cols)
df_pl = df_pl.loc[df_pl.index.isin(pair_long_keys)].reset_index()

idx_t1 = df_pl.groupby(grp_cols)['ses_num'].idxmin()
idx_tl = df_pl.groupby(grp_cols)['ses_num'].idxmax()

t1_p = df_pl.loc[idx_t1].set_index(grp_cols)
tl_p = df_pl.loc[idx_tl].set_index(grp_cols)

delta_pair = t1_p[['status', 'group', 'surgery_side']].copy()
delta_pair['delta_fisher_r'] = tl_p['fisher_r'] - t1_p['fisher_r']
delta_pair = delta_pair.reset_index()

# ── Bootstrap OTC vs Ctrl per category x pair ──
print()
print('=' * 80)
print('PAIRWISE SEARCHMASK (Longitudinal delta, bootstrap OTC v Ctrl)')
print('=' * 80)
print(f'{"Category":<12} {"Pair":<16} {"Ctrl dM(SD)":>14} {"OTC dM(SD)":>14} '
      f'{"OTC L-res":>10} {"OTC R-res":>10} {"p":>8}')
print('-' * 92)

pair_results = []
for cat in delta_pair['category'].unique():
    for pair_name in sorted(delta_pair[delta_pair['category'] == cat]['pair'].unique()):
        sub = delta_pair[(delta_pair['category'] == cat) & (delta_pair['pair'] == pair_name)]

        # Controls at preferred hemisphere
        pref = PREFERRED_CTRL_HEMI.get(cat, 'left')
        ctrl_v = sub[(sub['status'] == 'control') & (sub['hemi_label'] == pref)]['delta_fisher_r'].dropna().values

        # OTC at intact hemisphere
        otc_sub = sub[(sub['group'] == 'OTC') & (sub['hemi_label'] == 'intact')]
        otc_v = otc_sub['delta_fisher_r'].dropna().values

        otc_lr = otc_sub[otc_sub['surgery_side'] == 'left']['delta_fisher_r'].dropna().values
        otc_rr = otc_sub[otc_sub['surgery_side'] == 'right']['delta_fisher_r'].dropna().values

        p = bootstrap_p(otc_v, ctrl_v) if len(otc_v) > 0 and len(ctrl_v) > 0 else np.nan

        def m(a): return float(np.nanmean(a)) if len(a) > 0 else np.nan
        def s(a): return float(np.nanstd(a, ddof=1)) if len(a) > 1 else np.nan

        cm, cs = m(ctrl_v), s(ctrl_v)
        om, os_ = m(otc_v), s(otc_v)
        lr_m, rr_m = m(otc_lr), m(otc_rr)

        c_str = f'{cm:+.3f} ({cs:.3f})' if not np.isnan(cm) else chr(8212)
        o_str = f'{om:+.3f} ({os_:.3f})' if not np.isnan(om) else chr(8212)
        lr_str = f'{lr_m:+.3f}' if not np.isnan(lr_m) else chr(8212)
        rr_str = f'{rr_m:+.3f}' if not np.isnan(rr_m) else chr(8212)
        p_str = fmt_p(p) if 'fmt_p' in dir() else (f'{p:.3f}' if not np.isnan(p) else chr(8212))

        print(f'{cat:<12} {pair_name:<16} {c_str:>14} {o_str:>14} '
              f'{lr_str:>10} {rr_str:>10} {p_str:>8}')

        pair_results.append({
            'category': cat, 'pair': pair_name,
            'ctrl_delta_M': cm, 'ctrl_delta_SD': cs,
            'otc_delta_M': om, 'otc_delta_SD': os_,
            'otc_lresec': lr_m, 'otc_rresec': rr_m,
            'p_otc_v_ctrl': p,
        })

pair_df = pd.DataFrame(pair_results)
pair_df.to_csv('pairwise_searchmask_results.csv', index=False)
print('\nSaved: pairwise_searchmask_results.csv')

Pairwise columns: ['subject', 'subject_id', 'group', 'status', 'surgery_side', 'session', 'hemi', 'hemi_label', 'category', 'cat_type', 'roi_status', 'cope_set', 'pair', 'fisher_r']
Unique pairs: ['face-house' 'face-object' 'face-word' 'house-object' 'house-word'
 'object-word']
Unique categories: ['face' 'house' 'object' 'word' 'house_PPA' 'house_TOS' 'face_FFA'
 'object_LOC' 'object_pF' 'word_VWFA' 'word_STG' 'evc' 'face_STS']

PAIRWISE SEARCHMASK (Longitudinal delta, bootstrap OTC v Ctrl)
Category     Pair                Ctrl dM(SD)     OTC dM(SD)  OTC L-res  OTC R-res        p
--------------------------------------------------------------------------------------------
evc          face-house       -0.033 (1.016) -0.152 (0.976)     -0.261     +0.175    0.823
evc          face-object      +0.047 (0.474) +0.088 (1.054)     -0.223     +1.024    0.921
evc          face-word        +0.069 (0.728) +0.413 (0.802)     +0.240     +0.934    0.426
evc          house-object     -0.141 (0.779) -

In [44]:
# Cell 14: Within-OTC Unilateral vs Bilateral paired comparisons
# ═══════════════════════════════════════════════════════════════
# Geometry preservation and RDM distance: bilateral categories more
# disrupted than unilateral within each patient?
# Runs with and without sub-008.
# ═══════════════════════════════════════════════════════════════

from scipy.stats import ttest_rel

UNILATERAL = ['face', 'word']
BILATERAL  = ['house', 'object']

def get_uni_bi_per_patient(df, value_col, id_col='subject_id',
                           group_col='group', hemi_filter=True):
    '''Per OTC patient: mean of unilateral cats vs mean of bilateral cats.'''
    if hemi_filter and 'hemi_label' in df.columns:
        otc = df[(df[group_col] == 'OTC') & (df['hemi_label'] == 'intact')]
    else:
        otc = df[df[group_col] == 'OTC']
    otc = otc[otc['category'].isin(UNILATERAL + BILATERAL)]
    uni_means, bi_means, subs = [], [], []
    for sub in sorted(otc[id_col].unique()):
        sd = otc[otc[id_col] == sub]
        uni_vals = sd[sd['category'].isin(UNILATERAL)][value_col].dropna().values
        bi_vals  = sd[sd['category'].isin(BILATERAL)][value_col].dropna().values
        if len(uni_vals) > 0 and len(bi_vals) > 0:
            uni_means.append(np.mean(uni_vals))
            bi_means.append(np.mean(bi_vals))
            subs.append(sub)
    return np.array(uni_means), np.array(bi_means), subs


def get_uni_bi_ctrl(df, value_col, id_col='subject_id', hemi_col='hemi_label'):
    '''Same for controls at preferred hemisphere.'''
    ctrl_rows = []
    for cat in UNILATERAL + BILATERAL:
        pref = PREFERRED_CTRL_HEMI[cat]
        if hemi_col in df.columns:
            c = df[(df['status'] == 'control') & (df['category'] == cat) &
                   (df[hemi_col] == pref)]
        else:
            c = df[(df['group'] == 'control') & (df['category'] == cat)]
        for sub in c[id_col].unique():
            v = c[c[id_col] == sub][value_col].values
            if len(v) > 0:
                ctrl_rows.append({'sub': sub, 'category': cat, 'val': v[0]})
    cdf = pd.DataFrame(ctrl_rows)
    if len(cdf) == 0:
        return np.array([]), np.array([])
    uni_m, bi_m = [], []
    for sub in cdf['sub'].unique():
        sd = cdf[cdf['sub'] == sub]
        u = sd[sd['category'].isin(UNILATERAL)]['val'].values
        b = sd[sd['category'].isin(BILATERAL)]['val'].values
        if len(u) > 0 and len(b) > 0:
            uni_m.append(np.mean(u))
            bi_m.append(np.mean(b))
    return np.array(uni_m), np.array(bi_m)


def run_uni_bi_analysis(df, value_col, metric_name, id_col='subject_id',
                        hemi_col='hemi_label', higher_is_worse=True,
                        sub008_id='sub-008'):
    '''Full uni vs bi analysis with and without sub-008.'''
    print(f'\n{"=" * 70}')
    print(f'{metric_name}: UNILATERAL vs BILATERAL')
    print(f'{"=" * 70}')

    for label, exclude_extra in [('All OTC', []), ('Excl sub-008', [sub008_id])]:
        df_sub = df[~df[id_col].isin(exclude_extra)]
        uni, bi, subs = get_uni_bi_per_patient(
            df_sub, value_col, id_col=id_col
        )
        n = len(uni)
        if n < 2:
            print(f'\n  [{label}] n={n} -- too few for paired test')
            continue

        t, p = ttest_rel(uni, bi)
        print(f'\n  [{label}] n={n}')
        print(f'    Subjects: {subs}')
        print(f'    Uni M(SD): {np.mean(uni):.3f} ({np.std(uni, ddof=1):.3f})')
        print(f'    Bi  M(SD): {np.mean(bi):.3f} ({np.std(bi, ddof=1):.3f})')
        print(f'    Paired t({n-1}) = {t:.3f}, p = {p:.4f}', end='')
        print(' *' if p < .05 else '')

        print(f'    Per patient:')
        for i, sub in enumerate(subs):
            print(f'      {sub}: uni={uni[i]:.3f}, bi={bi[i]:.3f}, diff={uni[i]-bi[i]:+.3f}')

        # Bootstrap: OTC (uni-bi diff) vs control (uni-bi diff)
        uni_c, bi_c = get_uni_bi_ctrl(df_sub, value_col, id_col=id_col,
                                       hemi_col=hemi_col)
        if len(uni_c) > 0 and len(bi_c) > 0:
            ctrl_diff = uni_c - bi_c
            otc_diff = uni - bi
            p_boot = bootstrap_p(otc_diff, ctrl_diff)
            print(f'    Bootstrap OTC-diff vs Ctrl-diff: p = {p_boot:.4f}', end='')
            print(' *' if p_boot < .05 else '')
            print(f'    Ctrl uni-bi diff M(SD): {np.mean(ctrl_diff):.3f} ({np.std(ctrl_diff, ddof=1):.3f})')
            print(f'    OTC  uni-bi diff M(SD): {np.mean(otc_diff):.3f} ({np.std(otc_diff, ddof=1):.3f})')


# ── Geometry Preservation ──
run_uni_bi_analysis(df_geo, 'geometry_preservation', 'GEOMETRY PRESERVATION',
                    higher_is_worse=False)

# ── RDM Distance ──
run_uni_bi_analysis(df_rdm, 'rdm_distance', 'RDM DISTANCE',
                    id_col=rdm_id,
                    hemi_col='hemi_label' if 'hemi_label' in df_rdm.columns else 'none',
                    higher_is_worse=True)


GEOMETRY PRESERVATION: UNILATERAL vs BILATERAL

  [All OTC] n=5
    Subjects: ['sub-004', 'sub-008', 'sub-010', 'sub-021', 'sub-079']
    Uni M(SD): 0.671 (0.315)
    Bi  M(SD): 0.238 (0.272)
    Paired t(4) = 4.378, p = 0.0119 *
    Per patient:
      sub-004: uni=0.760, bi=0.223, diff=+0.537
      sub-008: uni=0.119, bi=-0.082, diff=+0.201
      sub-010: uni=0.808, bi=0.045, diff=+0.764
      sub-021: uni=0.748, bi=0.410, diff=+0.338
      sub-079: uni=0.917, bi=0.594, diff=+0.322
    Bootstrap OTC-diff vs Ctrl-diff: p = 0.1337
    Ctrl uni-bi diff M(SD): 0.249 (0.266)
    OTC  uni-bi diff M(SD): 0.432 (0.221)

  [Excl sub-008] n=4
    Subjects: ['sub-004', 'sub-010', 'sub-021', 'sub-079']
    Uni M(SD): 0.808 (0.077)
    Bi  M(SD): 0.318 (0.237)
    Paired t(3) = 4.743, p = 0.0178 *
    Per patient:
      sub-004: uni=0.760, bi=0.223, diff=+0.537
      sub-010: uni=0.808, bi=0.045, diff=+0.764
      sub-021: uni=0.748, bi=0.410, diff=+0.338
      sub-079: uni=0.917, bi=0.594, diff=

In [45]:
# Cell 15: Bootstrap CI — OTC group vs control distribution
# ═══════════════════════════════════════════════════════════════
# For each category x metric: bootstrap 95% CI of the control mean,
# then check whether OTC group mean falls within or outside.
# Also reports percentile rank of OTC mean in bootstrap distribution.
#
# Runs with and without sub-008 for longitudinal metrics.
# ═══════════════════════════════════════════════════════════════

def bootstrap_ci_comparison(ctrl_vals, otc_vals, metric_name, cat,
                            n_iter=N_ITER, rng=RNG, alpha=0.05):
    '''
    Bootstrap the control mean distribution and check where OTC mean falls.
    Returns: ctrl_mean, ctrl_ci_lo, ctrl_ci_hi, otc_mean, percentile, inside_ci
    '''
    ctrl_vals = ctrl_vals[~np.isnan(ctrl_vals)]
    otc_vals = otc_vals[~np.isnan(otc_vals)]
    if len(ctrl_vals) < 3 or len(otc_vals) == 0:
        return None

    boot_means = np.array([
        rng.choice(ctrl_vals, len(ctrl_vals), replace=True).mean()
        for _ in range(n_iter)
    ])
    ci_lo = np.percentile(boot_means, 100 * alpha / 2)
    ci_hi = np.percentile(boot_means, 100 * (1 - alpha / 2))
    otc_m = np.mean(otc_vals)
    pctile = np.mean(boot_means <= otc_m) * 100
    inside = ci_lo <= otc_m <= ci_hi

    return {
        'metric': metric_name, 'category': cat,
        'ctrl_M': np.mean(ctrl_vals), 'ctrl_SD': np.std(ctrl_vals, ddof=1),
        'ci_lo': ci_lo, 'ci_hi': ci_hi,
        'otc_M': otc_m, 'otc_SD': np.std(otc_vals, ddof=1) if len(otc_vals) > 1 else np.nan,
        'percentile': pctile, 'inside_ci': inside,
        'n_ctrl': len(ctrl_vals), 'n_otc': len(otc_vals),
    }


# ── Define all metric sources ──
# (df, value_col, metric_label, id_col, cross_sectional, schema)
# schema: 'geo' for geometry-style, 'sel' for selectivity-style

ci_datasets = [
    (df_geo, 'geometry_preservation', 'Geometry Preservation', 'subject_id', False, 'geo'),
]

# Add RDM if available
if 'df_rdm' in dir():
    ci_datasets.append(
        (df_rdm, 'rdm_distance', 'RDM Distance', rdm_id, False, 'rdm')
    )

# Add Liu cross-sectional
ci_datasets.append(
    (df_liu_cs, 'liu_distinctiveness', 'Liu Distinctiveness', 'subject_id', True, 'geo')
)

print('=' * 90)
print('BOOTSTRAP 95% CI: OTC mean vs Control distribution')
print('=' * 90)
print(f'{"Metric":<25} {"Cat":<8} {"Ctrl M":>8} {"95% CI":>18} '
      f'{"OTC M":>8} {"Pctile":>8} {"In CI?":>8}')
print('-' * 90)

ci_results = []

for df_src, vcol, mlabel, idcol, is_cs, schema in ci_datasets:
    for cat in CATEGORIES:
        if schema == 'geo':
            cv, ov, _, _, _ = extract_geo_schema(df_src, cat, vcol, cross_sectional=is_cs)
        elif schema == 'sel':
            cv, ov, _, _, _ = extract_sel_schema(df_src, cat, vcol)
        elif schema == 'rdm':
            c = df_src[df_src['category'] == cat]
            cv = c[c['group'] == 'control'][vcol].dropna().values
            ov = c[c['group'] == 'OTC'][vcol].dropna().values
        else:
            continue

        res = bootstrap_ci_comparison(cv, ov, mlabel, cat)
        if res is None:
            continue

        ci_results.append(res)
        sig = '' if res['inside_ci'] else ' ***'
        print(f'{mlabel:<25} {cat:<8} {res["ctrl_M"]:>8.3f} '
              f'[{res["ci_lo"]:>7.3f}, {res["ci_hi"]:>7.3f}] '
              f'{res["otc_M"]:>8.3f} {res["percentile"]:>7.1f}% '
              f'{"YES" if res["inside_ci"] else "NO":>6}{sig}')

ci_df = pd.DataFrame(ci_results)

# ── Highlight outside-CI results ──
outside = ci_df[~ci_df['inside_ci']]
if len(outside) > 0:
    print(f'\nRESULTS OUTSIDE 95% CI ({len(outside)}):')
    for _, r in outside.iterrows():
        direction = 'below' if r['otc_M'] < r['ci_lo'] else 'above'
        print(f'  {r["metric"]} / {r["category"]}: OTC={r["otc_M"]:.3f} '
              f'{direction} CI [{r["ci_lo"]:.3f}, {r["ci_hi"]:.3f}] '
              f'({r["percentile"]:.1f}th percentile)')
else:
    print('\nAll OTC means fall within control 95% CIs.')

ci_df.to_csv('bootstrap_ci_results.csv', index=False)
print('\nSaved: bootstrap_ci_results.csv')

BOOTSTRAP 95% CI: OTC mean vs Control distribution
Metric                    Cat        Ctrl M             95% CI    OTC M   Pctile   In CI?
------------------------------------------------------------------------------------------
Geometry Preservation     face        0.744 [  0.531,   0.901]    0.725    38.6%    YES
Geometry Preservation     house       0.399 [  0.068,   0.724]    0.080     2.9%    YES
Geometry Preservation     object      0.531 [  0.264,   0.731]    0.396    14.3%    YES
Geometry Preservation     word        0.767 [  0.620,   0.898]    0.542     0.2%     NO ***
RDM Distance              face        1.141 [  0.877,   1.428]    1.208    69.5%    YES
RDM Distance              house       1.730 [  1.209,   2.215]    2.116    93.8%    YES
RDM Distance              object      1.201 [  0.782,   1.685]    1.616    95.8%    YES
RDM Distance              word        1.255 [  0.879,   1.652]    1.611    96.0%    YES
Liu Distinctiveness       face        0.710 [  0.551,   0.87

In [46]:
# Cell 16: Sub-008 sensitivity analysis
# Re-run longitudinal bootstrap comparisons excluding sub-008.

SUB008 = 'sub-008'

print('=' * 100)
print('SUB-008 SENSITIVITY: Longitudinal metrics with vs without sub-008')
print('=' * 100)
print(f'{"Metric":<28} {"Cat":<8} {"p (all)":>10} {"p (excl 008)":>14} '
      f'{"OTC M (all)":>12} {"OTC M (no 008)":>16}')
print('-' * 100)

long_datasets = [
    (df_geo, 'geometry_preservation', 'Geometry Preservation', 'geo'),
]

if 'df_rdm' in dir():
    long_datasets.append((df_rdm, 'rdm_distance', 'RDM Distance', 'rdm'))

for df_src, vcol, mlabel, schema in long_datasets:
    for cat in CATEGORIES:
        if schema == 'geo':
            cv_all, ov_all, _, _, _ = extract_geo_schema(df_src, cat, vcol)
            id_col = 'subject_id'
        elif schema == 'rdm':
            c = df_src[df_src['category'] == cat]
            cv_all = c[c['group'] == 'control'][vcol].dropna().values
            ov_all = c[c['group'] == 'OTC'][vcol].dropna().values
            id_col = rdm_id

        p_all = bootstrap_p(ov_all, cv_all)
        otc_m_all = np.mean(ov_all) if len(ov_all) > 0 else np.nan

        df_no8 = df_src[~df_src[id_col].isin([SUB008])]
        if schema == 'geo':
            cv_no8, ov_no8, _, _, _ = extract_geo_schema(df_no8, cat, vcol)
        elif schema == 'rdm':
            c8 = df_no8[df_no8['category'] == cat]
            cv_no8 = c8[c8['group'] == 'control'][vcol].dropna().values
            ov_no8 = c8[c8['group'] == 'OTC'][vcol].dropna().values

        p_no8 = bootstrap_p(ov_no8, cv_no8)
        otc_m_no8 = np.mean(ov_no8) if len(ov_no8) > 0 else np.nan

        flag = ''
        if (p_all < .05) != (p_no8 < .05):
            flag = ' <-- CHANGED'
        elif abs(p_all - p_no8) > 0.1:
            flag = ' <-- shifted'

        print(f'{mlabel:<28} {cat:<8} {p_all:>10.4f} {p_no8:>14.4f} '
              f'{otc_m_all:>12.3f} {otc_m_no8:>16.3f}{flag}')

SUB-008 SENSITIVITY: Longitudinal metrics with vs without sub-008
Metric                       Cat         p (all)   p (excl 008)  OTC M (all)   OTC M (no 008)
----------------------------------------------------------------------------------------------------
Geometry Preservation        face         0.8943         0.3136        0.725            0.847 <-- shifted
Geometry Preservation        house        0.2130         0.3305        0.080            0.107 <-- shifted
Geometry Preservation        object       0.4426         0.9896        0.396            0.529 <-- shifted
Geometry Preservation        word         0.2044         0.6169        0.542            0.722 <-- shifted
RDM Distance                 face         0.8597         0.2694        1.208            0.890 <-- shifted
RDM Distance                 house        0.3714         0.8332        2.116            1.811 <-- shifted
RDM Distance                 object       0.1145         0.0957        1.616            1.668
RDM Dista

In [47]:
# Cell 17: Crawford-Howell per-patient tests
# Single-case test: each OTC patient vs control distribution.
# Crawford & Howell (1998): t = (x - M) / (s * sqrt((n+1)/n))

from scipy.stats import t as t_dist

def crawford_howell(patient_val, ctrl_vals):
    '''Crawford-Howell test for single case vs control group.'''
    ctrl_vals = ctrl_vals[~np.isnan(ctrl_vals)]
    if len(ctrl_vals) < 3 or np.isnan(patient_val):
        return np.nan, np.nan
    n = len(ctrl_vals)
    m = np.mean(ctrl_vals)
    s = np.std(ctrl_vals, ddof=1)
    if s == 0:
        return np.nan, np.nan
    t = (patient_val - m) / (s * np.sqrt((n + 1) / n))
    p = 2 * t_dist.sf(np.abs(t), df=n - 1)
    return float(t), float(p)


ch_datasets = [
    (df_geo, 'geometry_preservation', 'Geom Pres', 'geo', 'subject_id'),
]
if 'df_rdm' in dir():
    ch_datasets.append((df_rdm, 'rdm_distance', 'RDM Dist', 'rdm', rdm_id))
ch_datasets.append(
    (df_liu_cs, 'liu_distinctiveness', 'Liu Distinct', 'geo', 'subject_id')
)

print('=' * 95)
print('CRAWFORD-HOWELL PER-PATIENT TESTS')
print('=' * 95)

ch_results = []

for df_src, vcol, mlabel, schema, idcol in ch_datasets:
    print(f'\n--- {mlabel} ---')
    print(f'{"Patient":<12} {"Cat":<8} {"Value":>8} {"Ctrl M":>8} '
          f'{"Ctrl SD":>8} {"t":>8} {"p":>8}')
    print('-' * 70)

    for cat in CATEGORIES:
        if schema == 'geo':
            pref = PREFERRED_CTRL_HEMI[cat]
            c = df_src[df_src['category'] == cat]
            ctrl_v = c[(c['status'] == 'control') &
                       (c['hemi_label'] == pref)][vcol].dropna().values
            otc = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact')]
        elif schema == 'rdm':
            c = df_src[df_src['category'] == cat]
            ctrl_v = c[c['group'] == 'control'][vcol].dropna().values
            otc = c[c['group'] == 'OTC']

        for _, row in otc.iterrows():
            sub = row[idcol]
            val = row[vcol]
            if np.isnan(val):
                continue
            t_val, p_val = crawford_howell(val, ctrl_v)
            sig = '*' if p_val < .05 else ''
            sub_short = str(sub).replace('sub-', '')
            print(f'{sub_short:<12} {cat:<8} {val:>8.3f} {np.mean(ctrl_v):>8.3f} '
                  f'{np.std(ctrl_v, ddof=1):>8.3f} {t_val:>8.3f} {p_val:>8.4f} {sig}')

            ch_results.append({
                'metric': mlabel, 'subject': sub, 'category': cat,
                'patient_val': val, 'ctrl_M': np.mean(ctrl_v),
                'ctrl_SD': np.std(ctrl_v, ddof=1), 'n_ctrl': len(ctrl_v),
                't': t_val, 'p': p_val,
            })

ch_df = pd.DataFrame(ch_results)
sig_ch = ch_df[ch_df['p'] < .05]
print(f'\n{len(sig_ch)} significant Crawford-Howell results (p < .05):')
for _, r in sig_ch.iterrows():
    direction = 'below' if r['patient_val'] < r['ctrl_M'] else 'above'
    print(f'  {r["metric"]} / {r["subject"]} / {r["category"]}: '
          f'{r["patient_val"]:.3f} ({direction} ctrl M={r["ctrl_M"]:.3f}), '
          f't={r["t"]:.3f}, p={r["p"]:.4f}')

ch_df.to_csv('crawford_howell_results.csv', index=False)
print('\nSaved: crawford_howell_results.csv')

CRAWFORD-HOWELL PER-PATIENT TESTS

--- Geom Pres ---
Patient      Cat         Value   Ctrl M  Ctrl SD        t        p
----------------------------------------------------------------------
004          face        0.772    0.744    0.305    0.087   0.9324 
008          face        0.236    0.744    0.305   -1.580   0.1529 
010          face        0.787    0.744    0.305    0.134   0.8970 
021          face        0.911    0.744    0.305    0.519   0.6175 
079          face        0.917    0.744    0.305    0.538   0.6054 
004          house      -0.143    0.399    0.528   -0.975   0.3581 
008          house      -0.027    0.399    0.528   -0.765   0.4660 
010          house      -0.489    0.399    0.528   -1.596   0.1492 
021          house       0.253    0.399    0.528   -0.263   0.7991 
079          house       0.807    0.399    0.528    0.732   0.4848 
004          object      0.590    0.531    0.395    0.141   0.8913 
008          object     -0.137    0.531    0.395   -1.607   0

In [48]:
# Key Results At-a-Glance
# ═══════════════════════════════════════════════════════════════

print('=' * 95)
print('KEY RESULTS AT-A-GLANCE')
print('=' * 95)

# ── 1. Significant group-level bootstrap (from main table) ──
print('\n1. GROUP-LEVEL BOOTSTRAP (OTC vs Controls, main table)')
print('-' * 95)
print(f'{"Category":<10} {"Metric":<28} {"Ctrl M(SD)":>16} {"OTC M(SD)":>16} '
      f'{"Comparison":>16} {"p":>10}')
print('-' * 95)

rdf_sig = pd.DataFrame(results_rows)
sig_rows = []

# Collect all significant results
for _, r in rdf_sig.iterrows():
    for pcol, label in [('p OTC v Ctrl', 'OTC v Ctrl'),
                        ('p OTC v nonOTC', 'OTC v nonOTC'),
                        ('p nonOTC v Ctrl', 'nonOTC v Ctrl')]:
        p = r[pcol]
        if not np.isnan(p) and p < .05:
            sig_rows.append({
                'cat': r['Category'], 'metric': r['Metric'],
                'ctrl': f'{r["Ctrl M"]:.2f} ({r["Ctrl SD"]:.2f})' if not np.isnan(r['Ctrl SD']) else f'{r["Ctrl M"]:.2f}',
                'otc': f'{r["OTC M"]:.2f} ({r["OTC SD"]:.2f})' if not np.isnan(r['OTC SD']) else f'{r["OTC M"]:.2f}',
                'comp': label, 'p': p,
            })

for sr in sig_rows:
    p_str = '<.001*' if sr['p'] < .001 else f'{sr["p"]:.3f}*'
    print(f'{sr["cat"]:<10} {sr["metric"]:<28} {sr["ctrl"]:>16} {sr["otc"]:>16} '
          f'{sr["comp"]:>16} {p_str:>10}')

# ── 2. Bootstrap CI: OTC outside control 95% CI ──
print('\n2. OTC MEAN OUTSIDE CONTROL 95% CI')
print('-' * 95)
if 'ci_df' in dir() and len(ci_df) > 0:
    outside = ci_df[~ci_df['inside_ci']]
    if len(outside) > 0:
        for _, r in outside.iterrows():
            direction = 'BELOW' if r['otc_M'] < r['ci_lo'] else 'ABOVE'
            print(f'  {r["metric"]:<25} {r["category"]:<8} '
                  f'OTC={r["otc_M"]:.3f}  {direction} CI [{r["ci_lo"]:.3f}, {r["ci_hi"]:.3f}]  '
                  f'({r["percentile"]:.1f}th pctile)')
    else:
        print('  None')
else:
    print('  (Run Cell 15 first)')

# ── 3. Within-OTC uni vs bi ──
print('\n3. WITHIN-OTC UNILATERAL vs BILATERAL (paired t-tests)')
print('-' * 95)
print('  Geometry Preservation:')
print('    All OTC:       t(4)=4.378, p=.012 *   (uni=0.671 > bi=0.238)')
print('    Excl sub-008:  t(3)=4.743, p=.018 *   (uni=0.808 > bi=0.318)')
print('    Bootstrap diff-of-diff (excl 008): p=.049 *')
print('  RDM Distance:')
print('    All OTC:       t(4)=-2.155, p=.098     (bi=1.866 > uni=1.317)')
print('    Excl sub-008:  t(3)=-5.886, p=.010 *  (bi=1.740 > uni=0.957)')
print('    Bootstrap diff-of-diff (excl 008): p=.030 *')

# ── 4. Crawford-Howell significant individual patients ──
print('\n4. SIGNIFICANT CRAWFORD-HOWELL (individual patients vs controls)')
print('-' * 95)
if 'ch_df' in dir() and len(ch_df) > 0:
    sig_ch = ch_df[ch_df['p'] < .05].sort_values(['metric', 'category'])
    for _, r in sig_ch.iterrows():
        direction = 'below' if r['patient_val'] < r['ctrl_M'] else 'above'
        print(f'  {r["metric"]:<14} {str(r["subject"]):<10} {r["category"]:<8} '
              f'val={r["patient_val"]:.3f} ({direction} ctrl {r["ctrl_M"]:.3f})  '
              f't={r["t"]:.2f}, p={r["p"]:.4f}')
else:
    print('  (Run Cell 17 first)')

# ── 5. Sub-008 influence ──
print('\n5. SUB-008 INFLUENCE')
print('-' * 95)
print('  Sub-008 is individually significant (Crawford-Howell) on:')
print('    Geometry preservation / word:  t=-3.35, p=.012')
print('    MDS shift / face, object, word')
print('    RDM distance / face, word')
print('    Liu distinctiveness / word')
print('  Removing sub-008 strengthens uni-bi effects (geometry, RDM)')
print('  but no group-level results cross significance threshold either way.')

# ── 6. Trending results (p < .10) ──
print('\n6. TRENDING (p < .10, not significant)')
print('-' * 95)
for _, r in rdf_sig.iterrows():
    for pcol, label in [('p OTC v Ctrl', 'OTC v Ctrl'),
                        ('p OTC v nonOTC', 'OTC v nonOTC'),
                        ('p nonOTC v Ctrl', 'nonOTC v Ctrl')]:
        p = r[pcol]
        if not np.isnan(p) and .05 <= p < .10:
            print(f'  {r["Category"]:<10} {r["Metric"]:<28} {label:<16} p={p:.3f}')

print('\n' + '=' * 95)

KEY RESULTS AT-A-GLANCE

1. GROUP-LEVEL BOOTSTRAP (OTC vs Controls, main table)
-----------------------------------------------------------------------------------------------
Category   Metric                             Ctrl M(SD)        OTC M(SD)       Comparison          p
-----------------------------------------------------------------------------------------------
word       Peak Drift (mm)                   5.76 (5.90)     11.47 (4.19)       OTC v Ctrl     0.022*
face       Mean Activation                   4.71 (1.29)      4.24 (1.18)     OTC v nonOTC     0.021*
house      Volume                       4868.92 (2890.45) 3421.31 (1877.43)       OTC v Ctrl     0.046*
object     Volume                       20816.00 (7370.01) 12242.12 (7229.33)       OTC v Ctrl     <.001*
object     Volume                       20816.00 (7370.01) 12242.12 (7229.33)    nonOTC v Ctrl     0.041*
object     Sum Selectivity              1942.04 (865.54) 1253.14 (1039.04)       OTC v Ctrl     0.025*
wor

In [49]:
# FDR Correction (Benjamini-Hochberg)
# ═══════════════════════════════════════════════════════════════

def benjamini_hochberg(pvals):
    """Return FDR-corrected p-values (Benjamini-Hochberg)."""
    pvals = np.asarray(pvals)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = np.empty(n)
    ranked[order] = np.arange(1, n + 1)
    adjusted = pvals * n / ranked
    # Enforce monotonicity (step-down)
    adjusted = np.minimum.accumulate(adjusted[np.argsort(ranked)[::-1]])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    # Restore original order
    result = np.empty(n)
    result[np.argsort(ranked).astype(int)] = adjusted
    return result

rdf_fdr = pd.DataFrame(results_rows)

# Collect all p-values with their source info
all_tests = []
for _, r in rdf_fdr.iterrows():
    for pcol, label in [('p OTC v Ctrl', 'OTC v Ctrl'),
                        ('p OTC v nonOTC', 'OTC v nonOTC'),
                        ('p nonOTC v Ctrl', 'nonOTC v Ctrl')]:
        p = r[pcol]
        if not np.isnan(p):
            all_tests.append({
                'Category': r['Category'], 'Metric': r['Metric'],
                'Comparison': label, 'p_uncorr': p,
            })

fdr_df = pd.DataFrame(all_tests)

# Full-table FDR
fdr_df['p_fdr_all'] = benjamini_hochberg(fdr_df['p_uncorr'].values)

# Within-metric FDR
fdr_df['p_fdr_metric'] = np.nan
for metric in fdr_df['Metric'].unique():
    mask = fdr_df['Metric'] == metric
    fdr_df.loc[mask, 'p_fdr_metric'] = benjamini_hochberg(
        fdr_df.loc[mask, 'p_uncorr'].values)

# Print results
print('=' * 110)
print('FDR CORRECTION (Benjamini-Hochberg)')
print('=' * 110)

sig_uncorr = fdr_df[fdr_df['p_uncorr'] < .05].sort_values('p_uncorr')
print(f'\n{"Category":<10} {"Metric":<28} {"Comparison":<16} '
      f'{"p(uncorr)":>10} {"p(FDR-metric)":>14} {"p(FDR-all)":>12}')
print('-' * 95)

for _, r in sig_uncorr.iterrows():
    def star(p): return '*' if p < .05 else ''
    print(f'{r["Category"]:<10} {r["Metric"]:<28} {r["Comparison"]:<16} '
          f'{r["p_uncorr"]:>10.4f}* '
          f'{r["p_fdr_metric"]:>13.4f}{star(r["p_fdr_metric"])} '
          f'{r["p_fdr_all"]:>11.4f}{star(r["p_fdr_all"])}')

print(f'\nTotal tests: {len(fdr_df)}')
print(f'Significant uncorrected (p<.05): {(fdr_df["p_uncorr"] < .05).sum()}')
print(f'Significant FDR within-metric:   {(fdr_df["p_fdr_metric"] < .05).sum()}')
print(f'Significant FDR full-table:      {(fdr_df["p_fdr_all"] < .05).sum()}')

FDR CORRECTION (Benjamini-Hochberg)

Category   Metric                       Comparison        p(uncorr)  p(FDR-metric)   p(FDR-all)
-----------------------------------------------------------------------------------------------
object     Volume                       OTC v Ctrl           0.0001*        0.0012*      0.0068*
face       Mean Activation              OTC v nonOTC         0.0210*        0.2520      0.3679
word       Peak Drift (mm)              OTC v Ctrl           0.0221*        0.0884      0.3679
object     Sum Selectivity              OTC v Ctrl           0.0254*        0.2094      0.3679
house      Delta Sum Selectivity        OTC v Ctrl           0.0287*        0.1148      0.3679
word       Sum Selectivity              nonOTC v Ctrl        0.0349*        0.2094      0.3679
object     Volume                       nonOTC v Ctrl        0.0405*        0.1856      0.3679
house      Volume                       OTC v Ctrl           0.0464*        0.1856      0.3679

Total te

In [50]:
# ═══════════════════════════════════════════════════════════════════════════════
# DIFF-OF-DIFF: Two control hemisphere approaches
#   1) Preferred functional: face=RH, word=LH, house=RH, object=LH
#   2) Anatomically matched: match patient's intact hemisphere
# ═══════════════════════════════════════════════════════════════════════════════

from scipy.stats import ttest_rel
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path(processed_dir)
GEO_DIR = BASE / 'group_results' / 'geometry'
LIU_DIR = BASE / 'group_results' / 'liu_distinctiveness'
COPE_SET = 'differential'
EXCLUDE = ['sub-017']

CATEGORIES = ['face', 'house', 'object', 'word']
BILATERAL = ['house', 'object']
UNILATERAL = ['face', 'word']

PREFERRED_CTRL_HEMI = {
    'face': 'right', 'word': 'left', 'house': 'right', 'object': 'left',
}

N_BOOT = 100000
rng = np.random.default_rng(42)

# ── Load geometry ─────────────────────────────────────────────────────────
geo = pd.read_csv(GEO_DIR / f'geometry_{COPE_SET}.csv')
geo = geo[~geo['subject_id'].isin(EXCLUDE)]
geo = geo[geo['category'].isin(CATEGORIES)]

otc_geo = geo[(geo['group'] == 'OTC') & (geo['hemi_label'] == 'intact')]
ctrl_geo = geo[geo['status'] == 'control']

# ── Load pairwise for RDM distance ───────────────────────────────────────
pair_file = LIU_DIR / f'pairwise_correlations_{COPE_SET}.csv'
df_pw = pd.read_csv(pair_file)
df_pw = df_pw[~df_pw['subject_id'].isin(EXCLUDE)]
df_pw = df_pw[df_pw['category'].isin(CATEGORIES)]

ALL_PAIRS = sorted(df_pw['pair'].unique())

# Longitudinal only
ses_c = df_pw.groupby('subject_id')['session'].nunique()
multi = ses_c[ses_c >= 2].index.tolist()
df_pw_long = df_pw[df_pw['subject_id'].isin(multi)].copy()
df_pw_long['ses_rank'] = df_pw_long.groupby('subject_id')['session'].rank(
    method='dense').astype(int)
max_rank = df_pw_long.groupby('subject_id')['ses_rank'].transform('max')
df_pw_long = df_pw_long[(df_pw_long['ses_rank'] == 1) |
                         (df_pw_long['ses_rank'] == max_rank)].copy()
df_pw_long['tp'] = df_pw_long['ses_rank'].apply(lambda x: 'T1' if x == 1 else 'T2')

def compute_rdm_distance(df, subject_id, roi_cat):
    sub_df = df[df['subject_id'] == subject_id]
    t1_vals, t2_vals = [], []
    for pair in ALL_PAIRS:
        t1 = sub_df[(sub_df['category'] == roi_cat) &
                     (sub_df['pair'] == pair) &
                     (sub_df['tp'] == 'T1')]['fisher_r']
        t2 = sub_df[(sub_df['category'] == roi_cat) &
                     (sub_df['pair'] == pair) &
                     (sub_df['tp'] == 'T2')]['fisher_r']
        if len(t1) > 0 and len(t2) > 0:
            t1_vals.append(t1.values[0])
            t2_vals.append(t2.values[0])
    if len(t1_vals) < 6:
        return np.nan
    return np.sqrt(np.sum((np.array(t1_vals) - np.array(t2_vals))**2))


def get_otc_uni_bi(df, value_col, metric_name):
    """Get per-patient unilateral and bilateral means for OTC."""
    otc = df[(df['group'] == 'OTC') & (df['hemi_label'] == 'intact')]
    otc = otc[otc['category'].isin(CATEGORIES)]
    uni_m, bi_m, subs = [], [], []
    for sub in sorted(otc['subject_id'].unique()):
        sd = otc[otc['subject_id'] == sub]
        u = sd[sd['category'].isin(UNILATERAL)][value_col].dropna().values
        b = sd[sd['category'].isin(BILATERAL)][value_col].dropna().values
        if len(u) > 0 and len(b) > 0:
            uni_m.append(np.mean(u))
            bi_m.append(np.mean(b))
            subs.append(sub)
    return np.array(uni_m), np.array(bi_m), subs


def get_ctrl_uni_bi_preferred(df, value_col):
    """Controls at preferred hemisphere per category."""
    rows = []
    for sub in df[df['status'] == 'control']['subject_id'].unique():
        sd = df[(df['subject_id'] == sub) & (df['status'] == 'control')]
        u_vals, b_vals = [], []
        for cat in UNILATERAL:
            pref = PREFERRED_CTRL_HEMI[cat]
            v = sd[(sd['category'] == cat) & (sd['hemi_label'] == pref)][value_col].values
            if len(v) > 0:
                u_vals.append(v[0])
        for cat in BILATERAL:
            pref = PREFERRED_CTRL_HEMI[cat]
            v = sd[(sd['category'] == cat) & (sd['hemi_label'] == pref)][value_col].values
            if len(v) > 0:
                b_vals.append(v[0])
        if len(u_vals) > 0 and len(b_vals) > 0:
            rows.append({'sub': sub, 'uni': np.mean(u_vals), 'bi': np.mean(b_vals)})
    cdf = pd.DataFrame(rows)
    return cdf['uni'].values, cdf['bi'].values


def get_ctrl_uni_bi_matched(df, value_col, hemi_label):
    """Controls at one specific hemisphere for all categories."""
    rows = []
    for sub in df[df['status'] == 'control']['subject_id'].unique():
        sd = df[(df['subject_id'] == sub) & (df['status'] == 'control') &
                (df['hemi_label'] == hemi_label)]
        u_vals = sd[sd['category'].isin(UNILATERAL)][value_col].dropna().values
        b_vals = sd[sd['category'].isin(BILATERAL)][value_col].dropna().values
        if len(u_vals) > 0 and len(b_vals) > 0:
            rows.append({'sub': sub, 'uni': np.mean(u_vals), 'bi': np.mean(b_vals)})
    cdf = pd.DataFrame(rows)
    if len(cdf) == 0:
        return np.array([]), np.array([])
    return cdf['uni'].values, cdf['bi'].values


def run_diff_of_diff(otc_uni, otc_bi, ctrl_uni, ctrl_bi, label):
    """Run paired test + bootstrap diff-of-diff."""
    otc_diff = otc_uni - otc_bi  # for geometry: positive = uni better
    ctrl_diff = ctrl_uni - ctrl_bi

    print(f'\n  {label}:')
    print(f'    OTC:  uni M={np.mean(otc_uni):.3f}, bi M={np.mean(otc_bi):.3f}, '
          f'diff M={np.mean(otc_diff):.3f} (n={len(otc_diff)})')
    print(f'    Ctrl: uni M={np.mean(ctrl_uni):.3f}, bi M={np.mean(ctrl_bi):.3f}, '
          f'diff M={np.mean(ctrl_diff):.3f} (n={len(ctrl_diff)})')

    dd = np.mean(otc_diff) - np.mean(ctrl_diff)
    boot = np.empty(N_BOOT)
    for i in range(N_BOOT):
        o = rng.choice(otc_diff, size=len(otc_diff), replace=True)
        c = rng.choice(ctrl_diff, size=len(ctrl_diff), replace=True)
        boot[i] = o.mean() - c.mean()
    ci = np.percentile(boot, [2.5, 97.5])
    p = 2 * min(np.mean(boot <= 0), np.mean(boot >= 0))
    p = min(p, 1.0)
    sig = '*' if p < .05 else ''

    print(f'    Diff-of-diff: {dd:+.3f}, 95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}], p={p:.4f} {sig}')


# ══════════════════════════════════════════════════════════════════════════
# GEOMETRY PRESERVATION
# ══════════════════════════════════════════════════════════════════════════

print('='*70)
print('GEOMETRY PRESERVATION: Uni vs Bi diff-of-diff')
print('='*70)

otc_uni_g, otc_bi_g, subs_g = get_otc_uni_bi(geo, 'geometry_preservation', 'geo')

# Within OTC paired test
if len(otc_uni_g) >= 3:
    t, p = ttest_rel(otc_uni_g, otc_bi_g)
    print(f'\n  OTC within-patient: t({len(otc_uni_g)-1})={t:.3f}, p={p:.4f}')
    print(f'  Subjects: {subs_g}')

# Approach 1: Preferred functional hemisphere
ctrl_uni_pref, ctrl_bi_pref = get_ctrl_uni_bi_preferred(ctrl_geo, 'geometry_preservation')
run_diff_of_diff(otc_uni_g, otc_bi_g, ctrl_uni_pref, ctrl_bi_pref,
                 'Approach 1: Controls at PREFERRED hemisphere')

# Approach 2: Anatomically matched
# Split OTC by intact hemisphere, compare each to matched controls
for intact_hemi, hemi_label in [('left', 'left'), ('right', 'right')]:
    otc_hemi = geo[(geo['group'] == 'OTC') & (geo['hemi_label'] == 'intact') &
                    (geo['surgery_side'] != intact_hemi)]  # surgery opposite of intact
    # Actually just filter by intact hemisphere directly
    otc_hemi = geo[(geo['group'] == 'OTC') & (geo['hemi_label'] == 'intact')]
    # Get surgery_side to determine intact
    otc_intact = otc_hemi.copy()
    otc_intact['intact_hemi'] = otc_intact['surgery_side'].map(
        lambda s: 'right' if s == 'left' else 'left')
    otc_this = otc_intact[otc_intact['intact_hemi'] == intact_hemi]

    if len(otc_this['subject_id'].unique()) == 0:
        continue

    uni_m, bi_m, subs_h = [], [], []
    for sub in sorted(otc_this['subject_id'].unique()):
        sd = otc_this[otc_this['subject_id'] == sub]
        u = sd[sd['category'].isin(UNILATERAL)]['geometry_preservation'].dropna().values
        b = sd[sd['category'].isin(BILATERAL)]['geometry_preservation'].dropna().values
        if len(u) > 0 and len(b) > 0:
            uni_m.append(np.mean(u))
            bi_m.append(np.mean(b))
            subs_h.append(sub)

    ctrl_uni_m, ctrl_bi_m = get_ctrl_uni_bi_matched(
        ctrl_geo, 'geometry_preservation', hemi_label)

    if len(uni_m) > 0 and len(ctrl_uni_m) > 0:
        run_diff_of_diff(np.array(uni_m), np.array(bi_m),
                         ctrl_uni_m, ctrl_bi_m,
                         f'Approach 2: Controls at {hemi_label.upper()} (matched to intact {intact_hemi}), '
                         f'n_otc={len(uni_m)}, subs={subs_h}')

# ══════════════════════════════════════════════════════════════════════════
# RDM DISTANCE
# ══════════════════════════════════════════════════════════════════════════

print('\n' + '='*70)
print('RDM DISTANCE: Uni vs Bi diff-of-diff')
print('='*70)

# Compute RDM distance per OTC patient × category
otc_pw = df_pw_long[(df_pw_long['group'] == 'OTC') &
                     (df_pw_long['hemi_label'] == 'intact')]
ctrl_pw = df_pw_long[df_pw_long['status'] == 'control']

rdm_rows_otc = []
for sub in sorted(otc_pw['subject_id'].unique()):
    sub_code = otc_pw[otc_pw['subject_id'] == sub]['subject'].iloc[0]
    surgery = otc_pw[otc_pw['subject_id'] == sub]['surgery_side'].iloc[0]
    intact = 'right' if surgery == 'left' else 'left'
    for cat in CATEGORIES:
        d = compute_rdm_distance(otc_pw, sub, cat)
        if np.isfinite(d):
            rdm_rows_otc.append({
                'subject_id': sub, 'subject': sub_code,
                'intact_hemi': intact, 'category': cat,
                'cat_type': 'bilateral' if cat in BILATERAL else 'unilateral',
                'rdm_distance': d})

rdm_rows_ctrl = []
for sub in sorted(ctrl_pw['subject_id'].unique()):
    for hemi_label in ['left', 'right']:
        sub_hemi = ctrl_pw[(ctrl_pw['subject_id'] == sub) &
                            (ctrl_pw['hemi_label'] == hemi_label)]
        if len(sub_hemi) == 0:
            continue
        for cat in CATEGORIES:
            d = compute_rdm_distance(sub_hemi, sub, cat)
            if np.isfinite(d):
                rdm_rows_ctrl.append({
                    'subject_id': sub, 'hemi_label': hemi_label,
                    'category': cat,
                    'cat_type': 'bilateral' if cat in BILATERAL else 'unilateral',
                    'rdm_distance': d})

df_rdm_otc = pd.DataFrame(rdm_rows_otc)
df_rdm_ctrl = pd.DataFrame(rdm_rows_ctrl)

# OTC uni/bi
otc_rdm_uni, otc_rdm_bi, subs_r = [], [], []
for sub in df_rdm_otc['subject_id'].unique():
    sd = df_rdm_otc[df_rdm_otc['subject_id'] == sub]
    u = sd[sd['cat_type'] == 'unilateral']['rdm_distance'].mean()
    b = sd[sd['cat_type'] == 'bilateral']['rdm_distance'].mean()
    if np.isfinite(u) and np.isfinite(b):
        otc_rdm_uni.append(u)
        otc_rdm_bi.append(b)
        subs_r.append(sub)
otc_rdm_uni = np.array(otc_rdm_uni)
otc_rdm_bi = np.array(otc_rdm_bi)

if len(otc_rdm_uni) >= 3:
    t, p = ttest_rel(otc_rdm_uni, otc_rdm_bi)
    print(f'\n  OTC within-patient: t({len(otc_rdm_uni)-1})={t:.3f}, p={p:.4f}')
    print(f'  Subjects: {subs_r}')

# Approach 1: Preferred
ctrl_rdm_pref_rows = []
for sub in df_rdm_ctrl['subject_id'].unique():
    sd = df_rdm_ctrl[df_rdm_ctrl['subject_id'] == sub]
    u_vals, b_vals = [], []
    for cat in UNILATERAL:
        pref = PREFERRED_CTRL_HEMI[cat]
        v = sd[(sd['category'] == cat) & (sd['hemi_label'] == pref)]['rdm_distance'].values
        if len(v) > 0:
            u_vals.append(v[0])
    for cat in BILATERAL:
        pref = PREFERRED_CTRL_HEMI[cat]
        v = sd[(sd['category'] == cat) & (sd['hemi_label'] == pref)]['rdm_distance'].values
        if len(v) > 0:
            b_vals.append(v[0])
    if len(u_vals) > 0 and len(b_vals) > 0:
        ctrl_rdm_pref_rows.append({'uni': np.mean(u_vals), 'bi': np.mean(b_vals)})

cdf_pref = pd.DataFrame(ctrl_rdm_pref_rows)
run_diff_of_diff(otc_rdm_uni, otc_rdm_bi,
                 cdf_pref['uni'].values, cdf_pref['bi'].values,
                 'Approach 1: Controls at PREFERRED hemisphere')

# Approach 2: Anatomically matched
for intact_hemi in ['left', 'right']:
    otc_this = df_rdm_otc[df_rdm_otc['intact_hemi'] == intact_hemi]
    if len(otc_this['subject_id'].unique()) == 0:
        continue

    uni_m, bi_m, subs_h = [], [], []
    for sub in sorted(otc_this['subject_id'].unique()):
        sd = otc_this[otc_this['subject_id'] == sub]
        u = sd[sd['cat_type'] == 'unilateral']['rdm_distance'].mean()
        b = sd[sd['cat_type'] == 'bilateral']['rdm_distance'].mean()
        if np.isfinite(u) and np.isfinite(b):
            uni_m.append(u)
            bi_m.append(b)
            subs_h.append(sub)

    ctrl_hemi = df_rdm_ctrl[df_rdm_ctrl['hemi_label'] == intact_hemi]
    ctrl_uni_h, ctrl_bi_h = [], []
    for sub in ctrl_hemi['subject_id'].unique():
        sd = ctrl_hemi[ctrl_hemi['subject_id'] == sub]
        u = sd[sd['cat_type'] == 'unilateral']['rdm_distance'].mean()
        b = sd[sd['cat_type'] == 'bilateral']['rdm_distance'].mean()
        if np.isfinite(u) and np.isfinite(b):
            ctrl_uni_h.append(u)
            ctrl_bi_h.append(b)

    if len(uni_m) > 0 and len(ctrl_uni_h) > 0:
        run_diff_of_diff(np.array(uni_m), np.array(bi_m),
                         np.array(ctrl_uni_h), np.array(ctrl_bi_h),
                         f'Approach 2: Controls at {intact_hemi.upper()} '
                         f'(matched), n_otc={len(uni_m)}, subs={subs_h}')

# ══════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print('\n' + '='*70)
print('SUMMARY')
print('='*70)
print("""
Approach 1 (Preferred): Each control category uses its functionally
  preferred hemisphere (face=RH, word=LH, house=RH, object=LH).
  Controls get their "best" hemisphere per category.

Approach 2 (Anatomically matched): All control categories use the
  same hemisphere that the patient group has intact.
  L-resection patients (intact RH) → controls' RH for everything.
  R-resection patients (intact LH) → controls' LH for everything.
  This means word is compared against controls' RH (where word is weak).
""")

GEOMETRY PRESERVATION: Uni vs Bi diff-of-diff

  OTC within-patient: t(4)=4.378, p=0.0119
  Subjects: ['sub-004', 'sub-008', 'sub-010', 'sub-021', 'sub-079']

  Approach 1: Controls at PREFERRED hemisphere:
    OTC:  uni M=0.671, bi M=0.238, diff M=0.432 (n=5)
    Ctrl: uni M=0.714, bi M=0.465, diff M=0.249 (n=9)
    Diff-of-diff: +0.183, 95% CI [-0.050, +0.427], p=0.1278 

  Approach 2: Controls at LEFT (matched to intact left), n_otc=2, subs=['sub-004', 'sub-008']:
    OTC:  uni M=0.440, bi M=0.071, diff M=0.369 (n=2)
    Ctrl: uni M=0.702, bi M=0.509, diff M=0.193 (n=9)
    Diff-of-diff: +0.176, 95% CI [-0.184, +0.538], p=0.3495 

  Approach 2: Controls at RIGHT (matched to intact right), n_otc=3, subs=['sub-010', 'sub-021', 'sub-079']:
    OTC:  uni M=0.824, bi M=0.350, diff M=0.475 (n=3)
    Ctrl: uni M=0.486, bi M=0.518, diff M=-0.032 (n=9)
    Diff-of-diff: +0.507, 95% CI [+0.207, +0.831], p=0.0002 *

RDM DISTANCE: Uni vs Bi diff-of-diff

  OTC within-patient: t(4)=-2.155, p=0.0

In [51]:
# ═══════════════════════════════════════════════════════════════════════════════
# MASTER TABLE: Two control hemisphere approaches
# ═══════════════════════════════════════════════════════════════════════════════
# ── Load data if not already available ────────────────────────────────────
SEL_DIR = BASE / 'group_results' / 'selectivity'
sel_file = SEL_DIR / 'selectivity_summary.csv'
df_sel_all = pd.read_csv(sel_file)
df_sel_all = df_sel_all[~df_sel_all['sub'].isin(['sub-017'])]
df_sel_all['ses_int'] = df_sel_all['ses'].astype(int)
first = df_sel_all.groupby('sub')['ses_int'].min().reset_index().rename(columns={'ses_int': 'fs'})
df_sel_first = df_sel_all.merge(first, on='sub')
df_sel_first = df_sel_first[df_sel_first['ses_int'] == df_sel_first['fs']]

def fmt_msd(m, sd):
    if np.isnan(m): return chr(8212)
    if np.isnan(sd): return f'{m:.2f}'
    return f'{m:.2f} ({sd:.2f})'

def fmt_v(v):
    return chr(8212) if np.isnan(v) else f'{v:.2f}'

def fmt_p(p):
    if np.isnan(p): return chr(8212)
    if p < .001: return '<.001*'
    return f'{p:.3f}' + ('*' if p < .05 else '')

METRIC_ORDER = [
    'Peak Drift (mm)',
    'Mean Activation', 'Volume', 'Sum Selectivity',
    'Delta Sum Selectivity',
    'Liu Distinctiveness',
    'Delta Liu Distinctiveness',
    'Geometry Preservation',
    'MDS Shift',
    'RDM Distance',
]

# ═══════════════════════════════════════════════════════════════════════════
# APPROACH 1: Functionally Preferred Hemisphere
# (This is what the existing results_rows already contains)
# ═══════════════════════════════════════════════════════════════════════════

rdf1 = pd.DataFrame(results_rows)
cat_ord = {c: i for i, c in enumerate(CATEGORIES)}
met_ord = {m: i for i, m in enumerate(METRIC_ORDER)}
rdf1['_c'] = rdf1['Category'].map(cat_ord)
rdf1['_m'] = rdf1['Metric'].map(met_ord)
rdf1 = rdf1.sort_values(['_c', '_m']).drop(columns=['_c', '_m'])

rows1 = []
for _, r in rdf1.iterrows():
    rows1.append({
        'Category':      r['Category'].capitalize(),
        'Metric':        r['Metric'],
        'Ctrl M(SD)':    fmt_msd(r['Ctrl M'], r['Ctrl SD']),
        'OTC M(SD)':     fmt_msd(r['OTC M'], r['OTC SD']),
        'nonOTC M(SD)':  fmt_msd(r['nonOTC M'], r['nonOTC SD']),
        'OTC L-resec':   fmt_v(r['OTC L-resec']),
        'OTC R-resec':   fmt_v(r['OTC R-resec']),
        'OTC v Ctrl':    fmt_p(r['p OTC v Ctrl']),
        'OTC v nonOTC':  fmt_p(r['p OTC v nonOTC']),
        'nonOTC v Ctrl': fmt_p(r['p nonOTC v Ctrl']),
    })

display1 = pd.DataFrame(rows1)

print('='*120)
print('APPROACH 1: FUNCTIONALLY PREFERRED HEMISPHERE')
print('='*120)
print('Controls @ preferred hemi per category: face=RH, house=RH, object=LH, word=LH')
print('Patients @ intact hemisphere')
print('='*120)
print()
print(display1.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════
# APPROACH 2: Anatomically Matched Hemisphere
# Re-extract everything with controls matched to patient's intact side
# L-resection patients (intact RH) → controls RH
# R-resection patients (intact LH) → controls LH
# ═══════════════════════════════════════════════════════════════════════════

def extract_matched_geo(df, cat, value_col, ctrl_hemi, surgery_side, cross_sectional=False):
    """Extract values matching controls to a specific hemisphere."""
    c = df[df['category'] == cat]
    ctrl_vals = c[(c['status'] == 'control') & (c['hemi_label'] == ctrl_hemi)][value_col].dropna().values
    otc = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact') &
            (c['surgery_side'] == surgery_side)]
    otc_vals = otc[value_col].dropna().values
    nonotc_vals = np.array([])
    if cross_sectional:
        nonotc = c[(c['group'] == 'nonOTC') & (c['hemi_label'] == 'intact') &
                   (c['surgery_side'] == surgery_side)]
        nonotc_vals = nonotc[value_col].dropna().values
    return ctrl_vals, otc_vals, nonotc_vals

def extract_matched_sel(df, cat, value_col, ctrl_hemi, intact_hemi):
    """Extract selectivity values with controls at specific hemisphere."""
    c = df[df['category'] == cat]
    ctrl_vals = c[(c['group'] == 'control') & (c['hemi'] == ctrl_hemi)][value_col].dropna().values
    otc_all = c[c['group'] == 'OTC']
    otc = otc_all[(otc_all['hemi'] == otc_all['intact_hemi']) &
                   (otc_all['intact_hemi'] == intact_hemi)]
    otc_vals = otc[value_col].dropna().values
    non_all = c[c['group'] == 'nonOTC']
    non = non_all[(non_all['hemi'] == non_all['intact_hemi']) &
                   (non_all['intact_hemi'] == intact_hemi)]
    nonotc_vals = non[value_col].dropna().values
    return ctrl_vals, otc_vals, nonotc_vals

results_rows_matched = []

def add_row_matched(cat, metric, ctrl, otc, nonotc, p_oc, resec_label,
                    p_on=np.nan, p_nc=np.nan):
    def m(a): return float(np.nanmean(a)) if len(a) > 0 else np.nan
    def s(a): return float(np.nanstd(a, ddof=1)) if len(a) > 1 else np.nan
    results_rows_matched.append({
        'Category': cat, 'Metric': metric, 'Resection': resec_label,
        'Ctrl M': m(ctrl), 'Ctrl SD': s(ctrl),
        'OTC M': m(otc), 'OTC SD': s(otc),
        'nonOTC M': m(nonotc), 'nonOTC SD': s(nonotc),
        'p OTC v Ctrl': p_oc,
        'p OTC v nonOTC': p_on,
        'p nonOTC v Ctrl': p_nc,
    })

# ── Re-extract all measures with matched hemispheres ──────────────────────

# For each resection side: L resection (intact RH) vs controls RH
#                          R resection (intact LH) vs controls LH

for surgery_side, intact_hemi, ctrl_hemi, label in [
    ('left', 'right', 'right', 'L-resec (intact RH) vs Ctrl RH'),
    ('right', 'left', 'left', 'R-resec (intact LH) vs Ctrl LH'),
]:
    # Selectivity (cross-sectional)
    for metric_name, metric_col in [('Mean Activation', 'mean_act'),
                                     ('Volume', 'volume'),
                                     ('Sum Selectivity', 'sum_selec_norm')]:
        for cat in CATEGORIES:
            cv, ov, nv = extract_matched_sel(
                df_sel_first, cat, metric_col, ctrl_hemi, intact_hemi)
            p_oc = bootstrap_p(ov, cv) if len(ov) > 0 and len(cv) > 0 else np.nan
            p_nc = bootstrap_p(nv, cv) if len(nv) > 0 and len(cv) > 0 else np.nan
            p_on = bootstrap_p(ov, nv) if len(ov) > 0 and len(nv) > 0 else np.nan
            add_row_matched(cat, metric_name, cv, ov, nv, p_oc, label, p_on, p_nc)

    # Liu distinctiveness (cross-sectional)
    if 'df_liu' in dir() or 'df_liu' in globals():
        liu_cs = df_liu.copy() if 'df_liu' in dir() else globals()['df_liu'].copy()
        liu_cs = liu_cs[~liu_cs['subject_id'].isin(EXCLUDE)]
        liu_cs = liu_cs[liu_cs['category'].isin(CATEGORIES)]
        # First session
        if 'ses_num' not in liu_cs.columns:
            liu_cs['ses_num'] = liu_cs.groupby('subject_id')['session'].rank(
                method='dense').astype(int)
        liu_first = liu_cs[liu_cs['ses_num'] == 1]
        for cat in CATEGORIES:
            cv = liu_first[(liu_first['status'] == 'control') &
                           (liu_first['hemi_label'] == ctrl_hemi) &
                           (liu_first['category'] == cat)]['liu_distinctiveness'].dropna().values
            otc_cat = liu_first[(liu_first['group'] == 'OTC') &
                                (liu_first['hemi_label'] == 'intact') &
                                (liu_first['category'] == cat)]
            # Filter by surgery side
            ov = otc_cat[otc_cat['surgery_side'] == surgery_side]['liu_distinctiveness'].dropna().values
            p_oc = bootstrap_p(ov, cv) if len(ov) > 0 and len(cv) > 0 else np.nan
            add_row_matched(cat, 'Liu Distinctiveness', cv, ov, np.array([]), p_oc, label)

    # Geometry (longitudinal)
    for cat in CATEGORIES:
        c = df_geo[df_geo['category'] == cat]
        cv = c[(c['status'] == 'control') &
               (c['hemi_label'] == ctrl_hemi)]['geometry_preservation'].dropna().values
        otc_cat = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact') &
                     (c['surgery_side'] == surgery_side)]
        ov = otc_cat['geometry_preservation'].dropna().values
        p_oc = bootstrap_p(ov, cv) if len(ov) > 0 and len(cv) > 0 else np.nan
        add_row_matched(cat, 'Geometry Preservation', cv, ov, np.array([]), p_oc, label)

    # RDM Distance (longitudinal)
    for cat in CATEGORIES:
        cv = df_rdm_ctrl[df_rdm_ctrl['hemi_label'] == ctrl_hemi]
        cv = cv[cv['category'] == cat]['rdm_distance'].dropna().values
        ov = df_rdm_otc[(df_rdm_otc['intact_hemi'] == intact_hemi) &
                         (df_rdm_otc['category'] == cat)]['rdm_distance'].dropna().values
        p_oc = bootstrap_p(ov, cv) if len(ov) > 0 and len(cv) > 0 else np.nan
        add_row_matched(cat, 'RDM Distance', cv, ov, np.array([]), p_oc, label)

# Format approach 2
rdf2 = pd.DataFrame(results_rows_matched)
rdf2['_c'] = rdf2['Category'].map(cat_ord)
rdf2['_m'] = rdf2['Metric'].map(met_ord)
rdf2 = rdf2.sort_values(['Resection', '_c', '_m']).drop(columns=['_c', '_m'])

rows2 = []
for _, r in rdf2.iterrows():
    rows2.append({
        'Resection':     r['Resection'],
        'Category':      r['Category'].capitalize(),
        'Metric':        r['Metric'],
        'Ctrl M(SD)':    fmt_msd(r['Ctrl M'], r['Ctrl SD']),
        'OTC M(SD)':     fmt_msd(r['OTC M'], r['OTC SD']),
        'OTC v Ctrl':    fmt_p(r['p OTC v Ctrl']),
    })

display2 = pd.DataFrame(rows2)

print('\n\n')
print('='*120)
print('APPROACH 2: ANATOMICALLY MATCHED HEMISPHERE')
print('='*120)
print('L-resection patients (intact RH) compared to controls RH for ALL categories')
print('R-resection patients (intact LH) compared to controls LH for ALL categories')
print('='*120)

for resec_label in display2['Resection'].unique():
    print(f'\n--- {resec_label} ---')
    sub = display2[display2['Resection'] == resec_label].drop(columns=['Resection'])
    print(sub.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════
# Save both
# ═══════════════════════════════════════════════════════════════════════════

display1.to_csv('master_results_approach1_preferred.csv', index=False)
display2.to_csv('master_results_approach2_matched.csv', index=False)
print(f'\nSaved: master_results_approach1_preferred.csv')
print(f'Saved: master_results_approach2_matched.csv')

APPROACH 1: FUNCTIONALLY PREFERRED HEMISPHERE
Controls @ preferred hemi per category: face=RH, house=RH, object=LH, word=LH
Patients @ intact hemisphere

Category                    Metric         Ctrl M(SD)          OTC M(SD)       nonOTC M(SD) OTC L-resec OTC R-resec OTC v Ctrl OTC v nonOTC nonOTC v Ctrl
    Face           Peak Drift (mm)        4.50 (9.89)        3.92 (4.10)                  —        1.89        6.96      0.868            —             —
    Face           Mean Activation        4.71 (1.29)        4.24 (1.18)        5.08 (0.74)        4.61        3.87      0.223       0.021*         0.290
    Face                    Volume  1547.25 (1093.53)  1305.19 (1058.18)  2112.22 (1358.74)     1140.62     1469.75      0.478        0.102         0.242
    Face           Sum Selectivity    492.96 (429.66)    415.60 (338.26)    675.24 (434.40)      386.33      444.88      0.519        0.103         0.257
    Face     Delta Sum Selectivity    115.52 (179.07)    -61.33 (516.72)    

In [52]:
# Key Results At-a-Glance (Restructured)
# ═══════════════════════════════════════════════════════════════

print('=' * 100)
print('KEY RESULTS AT-A-GLANCE')
print('=' * 100)

print("""
═══════════════════════════════════════════════════════════════════════
A. LONGITUDINAL FINDINGS (n=5 OTC, 9 controls)
═══════════════════════════════════════════════════════════════════════

1. BILATERAL REPRESENTATIONAL DEGRADATION > UNILATERAL
───────────────────────────────────────────────────────────────────────
  Geometry preservation (uni vs bi within OTC):
    t(4) = 4.378, p = .012*   uni=0.671 > bi=0.238
    Bootstrap diff-of-diff vs controls: p = .126 †

  RDM distance (bi vs uni within OTC):
    t(4) = -2.155, p = .098   bi=1.866 > uni=1.317
    Bootstrap diff-of-diff vs controls: p = .320 †

  House delta sum selectivity (OTC vs Ctrl):
    Ctrl: +274 (296)  OTC: -125 (399)   p = .029*

  † Excl sub-008: geometry uni-bi p=.018*, diff-of-diff p=.049*;
    RDM distance uni-bi p=.010*, diff-of-diff p=.030*.
    Sub-008 (hemispherectomy) is the sole patient where unilateral
    exceeds bilateral on RDM distance. Leave-one-out identifies
    sub-008 as an outlier; all other patients show the expected
    bilateral > unilateral pattern.

2. UNILATERAL CATEGORIES SPATIALLY RELOCATE
───────────────────────────────────────────────────────────────────────
  Word peak drift (OTC vs Ctrl):
    Ctrl: 5.76 (5.90) mm  OTC: 11.47 (4.19) mm   p = .022*
    L-resec: 13.47 mm  R-resec: 8.47 mm
    No other category shows significant drift.

3. DISSOCIATION: DRIFT ≠ DEGRADATION
───────────────────────────────────────────────────────────────────────
  Spatial drift does not predict representational change:
    Drift ↔ Geometry preservation: rho = -.386, p = .156

4. PER-CATEGORY GROUP COMPARISONS (OTC vs Ctrl, bootstrap)
───────────────────────────────────────────────────────────────────────
  Geometry preservation:
    face .899  house .210  object .450  word .209
    Word falls below control 95% CI (0.2nd percentile)

  RDM distance:
    face .849  house .373  object .121  word .550

  Delta sum selectivity:
    face .419  house .029*  object .558  word .961

  Delta Liu distinctiveness:
    face .627  house .780  object .547  word .051

5. SIGNIFICANT INDIVIDUAL PATIENTS (Crawford-Howell)
───────────────────────────────────────────────────────────────────────
  sub-008:  Geom/word p=.012, RDM/face p=.022, RDM/word p=.028,
            Liu/word p=.033
  sub-021:  Liu/object p=.007, Liu/word p=.014
  sub-076:  Liu/word p=.012

═══════════════════════════════════════════════════════════════════════
B. CROSS-SECTIONAL SAMPLE CHARACTERIZATION
═══════════════════════════════════════════════════════════════════════
  (n=16 OTC, 9 nonOTC, 24 controls)

  Volume reduction (OTC vs Ctrl):
    object p < .001*   house p = .046*
    face n.s.          word n.s.

  Sum selectivity reduction (OTC vs Ctrl):
    object p = .025*
    All others n.s.

  Mean activation:
    OTC vs nonOTC face p = .021* (OTC lower)
    All OTC vs Ctrl n.s.

  Liu distinctiveness (OTC mean vs control 95% CI):
    house: OTC above CI (99.5th pctile)
    object: OTC above CI (99.0th pctile)
    word: OTC above CI (100th pctile)

  Mantel (RDM correlation):
    OTC-R vs Ctrl: r = .940, p = .040*
    OTC-L vs Ctrl: r = .420, p = .295

  nonOTC vs Ctrl:
    object volume p = .041*
    word sum selectivity p = .035*

═══════════════════════════════════════════════════════════════════════
""")

KEY RESULTS AT-A-GLANCE

═══════════════════════════════════════════════════════════════════════
A. LONGITUDINAL FINDINGS (n=5 OTC, 9 controls)
═══════════════════════════════════════════════════════════════════════

1. BILATERAL REPRESENTATIONAL DEGRADATION > UNILATERAL
───────────────────────────────────────────────────────────────────────
  Geometry preservation (uni vs bi within OTC):
    t(4) = 4.378, p = .012*   uni=0.671 > bi=0.238
    Bootstrap diff-of-diff vs controls: p = .126 †

  RDM distance (bi vs uni within OTC):
    t(4) = -2.155, p = .098   bi=1.866 > uni=1.317
    Bootstrap diff-of-diff vs controls: p = .320 †

  House delta sum selectivity (OTC vs Ctrl):
    Ctrl: +274 (296)  OTC: -125 (399)   p = .029*

  † Excl sub-008: geometry uni-bi p=.018*, diff-of-diff p=.049*;
    RDM distance uni-bi p=.010*, diff-of-diff p=.030*.
    Sub-008 (hemispherectomy) is the sole patient where unilateral
    exceeds bilateral on RDM distance. Leave-one-out identifies
    sub-008 as a

In [53]:
# Functional vs Anatomical Homolog Comparison
# ═══════════════════════════════════════════════════════════════
# Current notebook: OTC intact hemi vs controls' PREFERRED hemi (functional)
# This cell: OTC intact hemi vs controls' SAME hemi (anatomical)
#
# L-resec (n=3): sub-010, sub-021, sub-079 — intact RH vs Ctrl RH
# R-resec (n=2): sub-004, sub-008 — intact LH vs Ctrl LH
# ═══════════════════════════════════════════════════════════════

LRESEC = ['sub-010', 'sub-021', 'sub-079']  # intact RH
RRESEC = ['sub-004', 'sub-008']             # intact LH

# ── Helper: extract values with anatomical matching ──

def extract_anatomical(df, cat, value_col, patient_ids, ctrl_hemi,
                       schema='geo'):
    '''
    Get patient values (intact hemi) and control values at a
    SPECIFIC hemisphere (anatomical match), not preferred.
    '''
    c = df[df['category'] == cat]

    if schema == 'geo':
        ctrl = c[(c['status'] == 'control') & (c['hemi_label'] == ctrl_hemi)]
        ctrl_vals = ctrl[value_col].dropna().values
        otc = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact') &
                (c['subject_id'].isin(patient_ids))]
        otc_vals = otc[value_col].dropna().values
    elif schema == 'sel':
        ctrl = c[(c['group'] == 'control') & (c['hemi'] == ctrl_hemi)]
        ctrl_vals = ctrl[value_col].dropna().values
        otc_all = c[(c['group'] == 'OTC') & (c['sub'].isin(patient_ids))]
        otc = otc_all[otc_all['hemi'] == otc_all['intact_hemi']]
        otc_vals = otc[value_col].dropna().values
    elif schema == 'rdm':
        ctrl = c[c['group'] == 'control']
        ctrl_vals = ctrl[value_col].dropna().values  # RDM already averaged across hemis
        otc = c[(c['group'] == 'OTC') & (c['subject_id'].isin(patient_ids))]
        otc_vals = otc[value_col].dropna().values
    return ctrl_vals, otc_vals


# ── Uni vs Bi with anatomical matching (L-resec n=3 vs Ctrl RH) ──

print('=' * 95)
print('FUNCTIONAL vs ANATOMICAL HOMOLOG COMPARISON')
print('=' * 95)

print('\n─── L-RESEC (n=3, intact RH) vs CONTROLS RH ───')
print(f'Patients: {LRESEC}')

for metric_label, df_src, vcol, schema in [
    ('Geometry Preservation', df_geo, 'geometry_preservation', 'geo'),
    ('RDM Distance', df_rdm, 'rdm_distance', 'rdm'),
]:
    print(f'\n  {metric_label}:')
    print(f'    {"Category":<10} {"Ctrl RH M(SD)":>16} {"OTC M(SD)":>16} '
          f'{"p (anat)":>10} {"p (func)":>10}')
    print(f'    {"-"*68}')

    uni_otc, bi_otc = [], []
    uni_ctrl, bi_ctrl = [], []

    for cat in CATEGORIES:
        # Anatomical: ctrl RH
        cv_anat, ov = extract_anatomical(
            df_src, cat, vcol, LRESEC, 'right', schema=schema)
        p_anat = bootstrap_p(ov, cv_anat) if len(ov) > 0 and len(cv_anat) > 0 else np.nan

        # Functional: ctrl preferred (already computed in main table)
        pref = PREFERRED_CTRL_HEMI[cat]
        if schema == 'geo':
            cv_func, ov_func, _, _, _ = extract_geo_schema(df_src, cat, vcol)
        elif schema == 'rdm':
            c = df_src[df_src['category'] == cat]
            cv_func = c[c['group'] == 'control'][vcol].dropna().values
            ov_func = ov  # same OTC values

        p_func = bootstrap_p(ov, cv_func) if len(ov) > 0 and len(cv_func) > 0 else np.nan

        cm, cs = np.mean(cv_anat), np.std(cv_anat, ddof=1) if len(cv_anat) > 1 else np.nan
        om, os_ = np.mean(ov), np.std(ov, ddof=1) if len(ov) > 1 else np.nan

        def fmt_p(p):
            if np.isnan(p): return '—'
            return f'{p:.3f}*' if p < .05 else f'{p:.3f}'

        print(f'    {cat:<10} {cm:>7.3f} ({cs:.3f})  {om:>7.3f} ({os_:.3f})  '
              f'{fmt_p(p_anat):>10} {fmt_p(p_func):>10}')

        # Collect for uni-bi
        if cat in ['face', 'word']:
            if len(ov) > 0:
                for v in ov: uni_otc.append(v)
            for v in cv_anat: uni_ctrl.append(v)
        else:
            if len(ov) > 0:
                for v in ov: bi_otc.append(v)
            for v in cv_anat: bi_ctrl.append(v)

    # Uni vs bi within these L-resec patients
    uni_per, bi_per = [], []
    for sub in LRESEC:
        if schema == 'geo':
            sd = df_src[(df_src['subject_id'] == sub) &
                        (df_src['hemi_label'] == 'intact') &
                        (df_src['category'].isin(CATEGORIES))]
        elif schema == 'rdm':
            sd = df_src[(df_src['subject_id'] == sub) &
                        (df_src['category'].isin(CATEGORIES))]
        u = sd[sd['category'].isin(['face', 'word'])][vcol].mean()
        b = sd[sd['category'].isin(['house', 'object'])][vcol].mean()
        if np.isfinite(u) and np.isfinite(b):
            uni_per.append(u)
            bi_per.append(b)

    if len(uni_per) >= 2:
        t, p = ttest_rel(uni_per, bi_per)
        print(f'    Uni-bi paired (L-resec only): t({len(uni_per)-1})={t:.3f}, p={p:.4f}',
              end='')
        print(' *' if p < .05 else '')

    # Diff-of-diff: L-resec uni-bi diff vs ctrl RH uni-bi diff
    ctrl_uni_per, ctrl_bi_per = [], []
    for sub in df_src[df_src['status'] == 'control']['subject_id'].unique() if schema == 'geo' \
        else df_src[df_src['group'] == 'control']['subject_id'].unique():
        if schema == 'geo':
            sd = df_src[(df_src['subject_id'] == sub) &
                        (df_src['hemi_label'] == 'right') &
                        (df_src['category'].isin(CATEGORIES))]
        elif schema == 'rdm':
            sd = df_src[(df_src['subject_id'] == sub) &
                        (df_src['category'].isin(CATEGORIES))]
        u = sd[sd['category'].isin(['face', 'word'])][vcol].mean()
        b = sd[sd['category'].isin(['house', 'object'])][vcol].mean()
        if np.isfinite(u) and np.isfinite(b):
            ctrl_uni_per.append(u)
            ctrl_bi_per.append(b)

    otc_diffs = np.array(uni_per) - np.array(bi_per)
    ctrl_diffs = np.array(ctrl_uni_per) - np.array(ctrl_bi_per)
    if len(otc_diffs) > 0 and len(ctrl_diffs) > 0:
        p_dd = bootstrap_p(otc_diffs, ctrl_diffs)
        print(f'    Diff-of-diff (L-resec vs Ctrl RH): p={p_dd:.4f}',
              end='')
        print(' *' if p_dd < .05 else '')


# ── R-resec (n=2) — descriptive only ──
print('\n─── R-RESEC (n=2, intact LH) vs CONTROLS LH — descriptive only ───')
print(f'Patients: {RRESEC}')

for metric_label, df_src, vcol, schema in [
    ('Geometry Preservation', df_geo, 'geometry_preservation', 'geo'),
    ('RDM Distance', df_rdm, 'rdm_distance', 'rdm'),
]:
    print(f'\n  {metric_label}:')
    for cat in CATEGORIES:
        cv, ov = extract_anatomical(df_src, cat, vcol, RRESEC, 'left', schema=schema)
        if len(ov) > 0:
            print(f'    {cat:<10} Ctrl LH: {np.mean(cv):.3f} ({np.std(cv,ddof=1):.3f})  '
                  f'OTC: {", ".join(f"{v:.3f}" for v in ov)}')


# ── Summary table: functional vs anatomical p-values side by side ──
print('\n' + '=' * 95)
print('SUMMARY: Does anatomical matching change the conclusions?')
print('=' * 95)
print("""
For L-resec patients (n=3, intact RH):
  - Functional matching: OTC intact vs controls' preferred hemi
    (face=R, word=L, house=R, object=L)
  - Anatomical matching: OTC intact RH vs controls' RH for ALL categories

  Key difference: for word (preferred=LH), functional matching compares
  the patient's reorganized RH word region to controls' dominant LH word
  region. Anatomical matching compares it to controls' non-dominant RH.

  This matters for interpreting word results — are OTC word representations
  worse than the BEST control hemisphere, or worse than the SAME hemisphere?
""")

FUNCTIONAL vs ANATOMICAL HOMOLOG COMPARISON

─── L-RESEC (n=3, intact RH) vs CONTROLS RH ───
Patients: ['sub-010', 'sub-021', 'sub-079']

  Geometry Preservation:
    Category      Ctrl RH M(SD)        OTC M(SD)   p (anat)   p (func)
    --------------------------------------------------------------------
    face         0.744 (0.305)    0.871 (0.073)       0.210      0.215
    house        0.399 (0.528)    0.190 (0.650)       0.565      0.554
    object       0.637 (0.297)    0.509 (0.111)       0.225      0.877
    word        -0.087 (0.260)    0.708 (0.173)      0.000*      0.611
    Uni-bi paired (L-resec only): t(2)=3.285, p=0.0815
    Diff-of-diff (L-resec vs Ctrl RH): p=0.0008 *

  RDM Distance:
    Category      Ctrl RH M(SD)        OTC M(SD)   p (anat)   p (func)
    --------------------------------------------------------------------
    face         1.141 (0.448)    0.824 (0.474)       0.237      0.237
    house        1.730 (0.833)    1.590 (0.436)       0.677      0.685
 

In [54]:
# Key Results At-a-Glance (Final)
# ═══════════════════════════════════════════════════════════════

print("""
═══════════════════════════════════════════════════════════════════════════════════
KEY RESULTS AT-A-GLANCE
═══════════════════════════════════════════════════════════════════════════════════

A. LONGITUDINAL FINDINGS (n=5 OTC†, 9 controls)
─────────────────────────────────────────────────────────────────────────────────

1. BILATERAL REPRESENTATIONAL DEGRADATION > UNILATERAL
┌──────────────────────────────┬────────────────────────────────────────────────┐
│ Measure                      │ Result                                         │
├──────────────────────────────┼────────────────────────────────────────────────┤
│ Geometry pres (uni vs bi)    │ t(4)=4.378, p=.012*  uni=.671 > bi=.238 †    │
│   Diff-of-diff vs ctrl       │ p=.126 †                                      │
│ RDM distance (bi vs uni)     │ t(4)=-2.155, p=.098  bi=1.87 > uni=1.32 †    │
│   Diff-of-diff vs ctrl       │ p=.320 †                                      │
│ House Δ sum selectivity      │ Ctrl: +274  OTC: -125   p=.029*               │
│ Word geom pres vs ctrl CI    │ OTC=0.542, 0.2nd percentile (below 95% CI)    │
└──────────────────────────────┴────────────────────────────────────────────────┘

2. UNILATERAL CATEGORIES RELOCATE WITH FUNCTIONAL-LEVEL STABILITY
┌──────────────────────────────┬────────────────────────────────────────────────┐
│ Word peak drift (OTC v Ctrl) │ 11.47 vs 5.76 mm, p=.022*                    │
│   L-resec: 13.47  R-resec: 8.47 mm                                          │
│ No other category significant│                                               │
├──────────────────────────────┼────────────────────────────────────────────────┤
│ Word geom pres (anatomical)  │ OTC RH=.708 vs Ctrl RH=-.087, p<.001*        │
│   (functional comparison)    │ OTC=.708 vs Ctrl pref LH=.767, p=.611        │
│   Reorganized word achieves  │ dominant-hemisphere-level stability            │
└──────────────────────────────┴────────────────────────────────────────────────┘

3. UNI-BI DISSOCIATION UNDER ANATOMICAL MATCHING (L-resec n=3 vs Ctrl RH)
┌──────────────────────────────┬────────────────────────────────────────────────┐
│ Geom pres diff-of-diff       │ p=.0008*                                      │
│ Geom pres uni-bi (L-resec)   │ t(2)=3.285, p=.082                           │
│ RDM dist uni-bi (L-resec)    │ t(2)=-4.741, p=.042*                         │
│ RDM dist diff-of-diff        │ p=.069                                        │
└──────────────────────────────┴────────────────────────────────────────────────┘

4. DRIFT ≠ DEGRADATION
┌──────────────────────────────┬────────────────────────────────────────────────┐
│ Drift ↔ Geometry pres        │ rho=-.386, p=.156                             │
└──────────────────────────────┴────────────────────────────────────────────────┘

5. INDIVIDUAL PATIENTS (Crawford-Howell, p<.05)
┌──────────────────────────────┬────────────────────────────────────────────────┐
│ sub-008 †                    │ Geom/word p=.012, RDM/face p=.022,            │
│                              │ RDM/word p=.028, Liu/word p=.033              │
│ sub-021                      │ Liu/object p=.007, Liu/word p=.014            │
│ sub-076                      │ Liu/word p=.012                               │
└──────────────────────────────┴────────────────────────────────────────────────┘

† Sub-008 (hemispherectomy) is the sole patient where unilateral exceeds
  bilateral on RDM distance. Excluding sub-008:
    Geometry uni-bi: p=.018*, diff-of-diff: p=.049*
    RDM dist uni-bi: p=.010*, diff-of-diff: p=.030*

─────────────────────────────────────────────────────────────────────────────────
B. CROSS-SECTIONAL SAMPLE CHARACTERIZATION (16 OTC, 9 nonOTC, 24 controls)
─────────────────────────────────────────────────────────────────────────────────
┌──────────────────────────────┬────────────────────────────────────────────────┐
│ Volume (OTC v Ctrl)          │ object p<.001*, house p=.046*                 │
│ Sum selectivity (OTC v Ctrl) │ object p=.025*                               │
│ Mean act (OTC v nonOTC)      │ face p=.021*                                 │
│ Liu distinct vs ctrl 95% CI  │ house 99.5th, object 99.0th, word 100th pct  │
│ Mantel (RDM correlation)     │ OTC-R r=.940 p=.040*; OTC-L r=.420 p=.295   │
│ nonOTC v Ctrl                │ obj vol p=.041*, word selec p=.035*           │
└──────────────────────────────┴────────────────────────────────────────────────┘

═══════════════════════════════════════════════════════════════════════════════════
""")


═══════════════════════════════════════════════════════════════════════════════════
KEY RESULTS AT-A-GLANCE
═══════════════════════════════════════════════════════════════════════════════════

A. LONGITUDINAL FINDINGS (n=5 OTC†, 9 controls)
─────────────────────────────────────────────────────────────────────────────────

1. BILATERAL REPRESENTATIONAL DEGRADATION > UNILATERAL
┌──────────────────────────────┬────────────────────────────────────────────────┐
│ Measure                      │ Result                                         │
├──────────────────────────────┼────────────────────────────────────────────────┤
│ Geometry pres (uni vs bi)    │ t(4)=4.378, p=.012*  uni=.671 > bi=.238 †    │
│   Diff-of-diff vs ctrl       │ p=.126 †                                      │
│ RDM distance (bi vs uni)     │ t(4)=-2.155, p=.098  bi=1.87 > uni=1.32 †    │
│   Diff-of-diff vs ctrl       │ p=.320 †                                      │
│ House Δ sum selectivity      │ Ctrl: +274  OTC: -1

In [55]:
# Extract peak drift by hemisphere for all groups
grp = ['sub', 'category', 'hemi']
idx_first = df_peak.groupby(grp)['ses_num'].idxmin()
idx_last  = df_peak.groupby(grp)['ses_num'].idxmax()

t1 = df_peak.loc[idx_first].set_index(grp)
tl = df_peak.loc[idx_last].set_index(grp)
multi = t1.index[t1['ses_num'] != tl.loc[t1.index, 'ses_num']]
t1, tl = t1.loc[multi], tl.loc[multi]

drift = pd.DataFrame(index=multi)
drift['peak_drift_mm'] = np.sqrt(
    (tl['peak_x_mni'] - t1['peak_x_mni'])**2 +
    (tl['peak_y_mni'] - t1['peak_y_mni'])**2 +
    (tl['peak_z_mni'] - t1['peak_z_mni'])**2
)
drift['group'] = t1['group']
drift['intact_hemi'] = t1['intact_hemi']
drift = drift.reset_index()

for cat in CATEGORIES:
    print(f'\n{cat}:')
    for hemi in ['left', 'right']:
        ctrl = drift[(drift['group'] == 'control') & 
                     (drift['hemi'] == hemi) & 
                     (drift['category'] == cat)]['peak_drift_mm']
        if len(ctrl) > 0:
            print(f'  Ctrl {hemi}: M={ctrl.mean():.2f}, SD={ctrl.std():.2f}, n={len(ctrl)}')
    
    otc = drift[(drift['group'] == 'OTC') & (drift['category'] == cat)]
    for side in ['left', 'right']:
        sub = otc[otc['intact_hemi'] == side]['peak_drift_mm']
        label = 'L-resec (intact RH)' if side == 'right' else 'R-resec (intact LH)'
        if len(sub) > 0:
            print(f'  {label}: M={sub.mean():.2f}, SD={sub.std():.2f}, n={len(sub)}, vals={sub.values.round(2)}')


face:
  Ctrl left: M=6.08, SD=6.71, n=9
  Ctrl right: M=4.50, SD=9.89, n=9
  R-resec (intact LH): M=6.96, SD=5.40, n=2, vals=[ 3.14 10.77]
  L-resec (intact RH): M=1.89, SD=1.93, n=3, vals=[4.11 0.61 0.95]

house:
  Ctrl left: M=7.46, SD=10.65, n=9
  Ctrl right: M=10.29, SD=10.01, n=9
  R-resec (intact LH): M=18.80, SD=3.38, n=2, vals=[21.19 16.41]
  L-resec (intact RH): M=10.73, SD=10.57, n=3, vals=[ 8.74 22.15  1.3 ]

object:
  Ctrl left: M=4.55, SD=4.03, n=9
  Ctrl right: M=5.24, SD=4.34, n=9
  R-resec (intact LH): M=10.57, SD=7.77, n=2, vals=[ 5.08 16.06]
  L-resec (intact RH): M=3.67, SD=4.09, n=3, vals=[8.31 0.61 2.08]

word:
  Ctrl left: M=5.76, SD=5.90, n=8
  Ctrl right: M=13.83, SD=10.39, n=8
  R-resec (intact LH): M=8.47, SD=0.41, n=2, vals=[8.17 8.76]
  L-resec (intact RH): M=13.47, SD=4.48, n=3, vals=[13.26 18.05  9.1 ]


In [56]:
# ── Selectivity by hemisphere (cross-sectional, first session) ──
SELEC_METRICS = {'mean_act': 'Mean Activation', 'volume': 'Volume', 'sum_selec_norm': 'Sum Selectivity'}

for col, label in SELEC_METRICS.items():
    print(f'\n{"="*70}')
    print(f'{label}')
    print(f'{"="*70}')
    for cat in CATEGORIES:
        print(f'\n  {cat}:')
        for hemi in ['left', 'right']:
            ctrl = df_cs[(df_cs['group'] == 'control') & 
                         (df_cs['category'] == cat) &
                         (df_cs['hemi'] == hemi)][col]
            if len(ctrl) > 0:
                print(f'    Ctrl {hemi}: M={ctrl.mean():.2f}, SD={ctrl.std():.2f}, n={len(ctrl)}')
        
        otc = df_cs[(df_cs['group'] == 'OTC') & (df_cs['category'] == cat)]
        for intact in ['left', 'right']:
            sub = otc[otc['intact_hemi'] == intact]
            sub_v = sub[sub['hemi'] == intact][col]
            label_r = 'R-resec (intact LH)' if intact == 'left' else 'L-resec (intact RH)'
            if len(sub_v) > 0:
                print(f'    {label_r}: M={sub_v.mean():.2f}, SD={sub_v.std():.2f}, n={len(sub_v)}')
        
        nonotc = df_cs[(df_cs['group'] == 'nonOTC') & (df_cs['category'] == cat)]
        for intact in ['left', 'right']:
            sub_n = nonotc[nonotc['intact_hemi'] == intact]
            sub_nv = sub_n[sub_n['hemi'] == intact][col]
            label_n = 'nonOTC intact LH' if intact == 'left' else 'nonOTC intact RH'
            if len(sub_nv) > 0:
                print(f'    {label_n}: M={sub_nv.mean():.2f}, SD={sub_nv.std():.2f}, n={len(sub_nv)}')


Mean Activation

  face:
    Ctrl left: M=4.37, SD=1.35, n=24
    Ctrl right: M=4.71, SD=1.29, n=24
    R-resec (intact LH): M=3.87, SD=0.99, n=8
    L-resec (intact RH): M=4.61, SD=1.30, n=8
    nonOTC intact LH: M=5.12, SD=0.94, n=5
    nonOTC intact RH: M=5.04, SD=0.54, n=4

  house:
    Ctrl left: M=4.18, SD=0.55, n=24
    Ctrl right: M=4.22, SD=0.74, n=24
    R-resec (intact LH): M=3.76, SD=0.82, n=8
    L-resec (intact RH): M=4.16, SD=0.84, n=8
    nonOTC intact LH: M=4.26, SD=0.94, n=5
    nonOTC intact RH: M=4.81, SD=1.15, n=4

  object:
    Ctrl left: M=4.74, SD=0.86, n=24
    Ctrl right: M=4.41, SD=0.72, n=24
    R-resec (intact LH): M=4.02, SD=0.96, n=8
    L-resec (intact RH): M=4.54, SD=1.09, n=8
    nonOTC intact LH: M=5.16, SD=1.18, n=5
    nonOTC intact RH: M=4.63, SD=0.53, n=4

  word:
    Ctrl left: M=3.41, SD=0.75, n=24
    Ctrl right: M=2.94, SD=0.52, n=24
    R-resec (intact LH): M=3.04, SD=0.82, n=8
    L-resec (intact RH): M=3.07, SD=0.54, n=8
    nonOTC intact 

In [57]:
# ── Delta Selectivity by hemisphere (longitudinal) ──
print('='*70)
print('Delta Sum Selectivity (T_last - T1)')
print('='*70)

for cat in CATEGORIES:
    print(f'\n  {cat}:')
    for hemi in ['left', 'right']:
        ctrl = delta_sel[(delta_sel['group'] == 'control') &
                         (delta_sel['category'] == cat) &
                         (delta_sel['hemi'] == hemi)]['delta_ssn']
        if len(ctrl) > 0:
            print(f'    Ctrl {hemi}: M={ctrl.mean():.2f}, SD={ctrl.std():.2f}, n={len(ctrl)}')
    
    otc = delta_sel[(delta_sel['group'] == 'OTC') & (delta_sel['category'] == cat)]
    for intact in ['left', 'right']:
        sub = otc[otc['intact_hemi'] == intact]
        sub_v = sub[sub['hemi'] == intact]['delta_ssn']
        label_r = 'R-resec (intact LH)' if intact == 'left' else 'L-resec (intact RH)'
        if len(sub_v) > 0:
            print(f'    {label_r}: M={sub_v.mean():.2f}, SD={sub_v.std():.2f}, n={len(sub_v)}, vals={sub_v.values.round(2)}')

Delta Sum Selectivity (T_last - T1)

  face:
    Ctrl left: M=42.81, SD=131.30, n=9
    Ctrl right: M=115.52, SD=179.07, n=9
    R-resec (intact LH): M=-207.65, SD=882.16, n=2, vals=[ 416.13 -831.44]
    L-resec (intact RH): M=36.22, SD=330.47, n=3, vals=[  -2.06  384.15 -273.45]

  house:
    Ctrl left: M=220.29, SD=380.44, n=9
    Ctrl right: M=274.14, SD=296.02, n=9
    R-resec (intact LH): M=-72.64, SD=684.10, n=2, vals=[ 411.1  -556.37]
    L-resec (intact RH): M=-160.11, SD=283.07, n=3, vals=[-210.    144.59 -414.92]

  object:
    Ctrl left: M=-120.22, SD=1289.72, n=9
    Ctrl right: M=-232.96, SD=1257.01, n=9
    R-resec (intact LH): M=26.27, SD=1021.48, n=2, vals=[ 748.57 -696.02]
    L-resec (intact RH): M=282.01, SD=823.43, n=3, vals=[-605.29  429.75 1021.57]

  word:
    Ctrl left: M=22.23, SD=170.05, n=9
    Ctrl right: M=9.61, SD=34.97, n=9
    R-resec (intact LH): M=46.22, SD=68.00, n=2, vals=[94.3  -1.86]
    L-resec (intact RH): M=11.44, SD=18.91, n=3, vals=[-6.97 30.8

In [58]:
# ── Liu Distinctiveness by hemisphere (cross-sectional) ──
print('='*70)
print('Liu Distinctiveness (cross-sectional)')
print('='*70)

for cat in CATEGORIES:
    print(f'\n  {cat}:')
    for hemi_label in ['left', 'right']:
        ctrl = df_liu_cs[(df_liu_cs['status'] == 'control') &
                         (df_liu_cs['category'] == cat) &
                         (df_liu_cs['hemi_label'] == hemi_label)]['liu_distinctiveness']
        if len(ctrl) > 0:
            print(f'    Ctrl {hemi_label}: M={ctrl.mean():.2f}, SD={ctrl.std():.2f}, n={len(ctrl)}')
    
    otc = df_liu_cs[(df_liu_cs['group'] == 'OTC') & (df_liu_cs['category'] == cat)]
    for hl in ['intact']:
        sub = otc[otc['hemi_label'] == hl]['liu_distinctiveness']
        if len(sub) > 0:
            # Split by surgery side
            for ss in ['left', 'right']:
                ss_sub = otc[(otc['hemi_label'] == hl) & (otc['surgery_side'] == ss)]['liu_distinctiveness']
                label_r = 'L-resec (intact RH)' if ss == 'left' else 'R-resec (intact LH)'
                if len(ss_sub) > 0:
                    print(f'    {label_r}: M={ss_sub.mean():.2f}, SD={ss_sub.std():.2f}, n={len(ss_sub)}')
    
    nonotc = df_liu_cs[(df_liu_cs['group'] == 'nonOTC') & (df_liu_cs['category'] == cat)]
    for hl in ['intact']:
        sub_n = nonotc[nonotc['hemi_label'] == hl]['liu_distinctiveness']
        if len(sub_n) > 0:
            print(f'    nonOTC intact: M={sub_n.mean():.2f}, SD={sub_n.std():.2f}, n={len(sub_n)}')

Liu Distinctiveness (cross-sectional)

  face:
    Ctrl left: M=0.71, SD=0.37, n=21
    Ctrl right: M=0.71, SD=0.39, n=21
    L-resec (intact RH): M=0.85, SD=0.25, n=7
    R-resec (intact LH): M=0.75, SD=0.33, n=7
    nonOTC intact: M=0.66, SD=0.24, n=9

  house:
    Ctrl left: M=0.48, SD=0.81, n=22
    Ctrl right: M=0.11, SD=0.81, n=22
    L-resec (intact RH): M=0.48, SD=0.69, n=7
    R-resec (intact LH): M=0.61, SD=0.66, n=8
    nonOTC intact: M=0.33, SD=0.59, n=9

  object:
    Ctrl left: M=1.03, SD=0.33, n=22
    Ctrl right: M=1.17, SD=0.53, n=22
    L-resec (intact RH): M=1.30, SD=0.49, n=7
    R-resec (intact LH): M=1.09, SD=0.37, n=8
    nonOTC intact: M=1.22, SD=0.32, n=9

  word:
    Ctrl left: M=0.51, SD=0.36, n=21
    Ctrl right: M=0.39, SD=0.48, n=16
    L-resec (intact RH): M=0.75, SD=0.42, n=6
    R-resec (intact LH): M=0.80, SD=0.60, n=7
    nonOTC intact: M=0.79, SD=0.47, n=9


In [59]:
# ── Delta Liu Distinctiveness by hemisphere (longitudinal) ──
print('='*70)
print('Delta Liu Distinctiveness (T_last - T1)')
print('='*70)

for cat in CATEGORIES:
    print(f'\n  {cat}:')
    for hemi_label in ['left', 'right']:
        ctrl = delta_liu[(delta_liu['status'] == 'control') &
                         (delta_liu['category'] == cat) &
                         (delta_liu['hemi_label'] == hemi_label)]['delta_liu']
        if len(ctrl) > 0:
            print(f'    Ctrl {hemi_label}: M={ctrl.mean():.2f}, SD={ctrl.std():.2f}, n={len(ctrl)}')
    
    otc = delta_liu[(delta_liu['group'] == 'OTC') & (delta_liu['category'] == cat)]
    for ss in ['left', 'right']:
        sub = otc[(otc['hemi_label'] == 'intact') & (otc['surgery_side'] == ss)]['delta_liu']
        label_r = 'L-resec (intact RH)' if ss == 'left' else 'R-resec (intact LH)'
        if len(sub) > 0:
            print(f'    {label_r}: M={sub.mean():.2f}, SD={sub.std():.2f}, n={len(sub)}, vals={sub.values.round(2)}')

Delta Liu Distinctiveness (T_last - T1)

  face:
    Ctrl left: M=0.18, SD=0.42, n=9
    Ctrl right: M=-0.11, SD=0.41, n=9
    L-resec (intact RH): M=0.03, SD=0.35, n=3, vals=[ 0.43 -0.11 -0.22]
    R-resec (intact LH): M=-0.59, SD=0.27, n=2, vals=[-0.4  -0.78]

  house:
    Ctrl left: M=-0.23, SD=0.64, n=9
    Ctrl right: M=-0.34, SD=0.99, n=9
    L-resec (intact RH): M=-0.00, SD=0.35, n=3, vals=[-0.34 -0.02  0.35]
    R-resec (intact LH): M=-1.13, SD=0.11, n=2, vals=[-1.06 -1.21]

  object:
    Ctrl left: M=0.12, SD=0.55, n=9
    Ctrl right: M=0.09, SD=0.53, n=9
    L-resec (intact RH): M=-0.00, SD=0.59, n=3, vals=[ 0.59 -0.58 -0.02]
    R-resec (intact LH): M=-0.15, SD=0.85, n=2, vals=[ 0.46 -0.75]

  word:
    Ctrl left: M=0.21, SD=0.38, n=8
    Ctrl right: M=-0.20, SD=0.41, n=5
    L-resec (intact RH): M=-0.41, SD=0.75, n=2, vals=[ 0.12 -0.94]
    R-resec (intact LH): M=-0.56, SD=1.08, n=2, vals=[ 0.21 -1.32]


In [60]:
# ── Geometry Preservation by hemisphere (longitudinal) ──
print('='*70)
print('Geometry Preservation')
print('='*70)

for cat in CATEGORIES:
    print(f'\n  {cat}:')
    for hemi_label in ['left', 'right']:
        ctrl = df_geo[(df_geo['status'] == 'control') &
                      (df_geo['category'] == cat) &
                      (df_geo['hemi_label'] == hemi_label)]['geometry_preservation']
        if len(ctrl) > 0:
            print(f'    Ctrl {hemi_label}: M={ctrl.mean():.2f}, SD={ctrl.std():.2f}, n={len(ctrl)}')
    
    otc = df_geo[(df_geo['group'] == 'OTC') & (df_geo['category'] == cat)]
    for ss in ['left', 'right']:
        sub = otc[(otc['hemi_label'] == 'intact') & (otc['surgery_side'] == ss)]['geometry_preservation']
        label_r = 'L-resec (intact RH)' if ss == 'left' else 'R-resec (intact LH)'
        if len(sub) > 0:
            print(f'    {label_r}: M={sub.mean():.2f}, SD={sub.std():.2f}, n={len(sub)}, vals={sub.values.round(2)}')

Geometry Preservation

  face:
    Ctrl left: M=0.65, SD=0.25, n=9
    Ctrl right: M=0.74, SD=0.30, n=9
    L-resec (intact RH): M=0.87, SD=0.07, n=3, vals=[0.79 0.91 0.92]
    R-resec (intact LH): M=0.50, SD=0.38, n=2, vals=[0.77 0.24]

  house:
    Ctrl left: M=0.49, SD=0.60, n=9
    Ctrl right: M=0.40, SD=0.53, n=9
    L-resec (intact RH): M=0.19, SD=0.65, n=3, vals=[-0.49  0.25  0.81]
    R-resec (intact LH): M=-0.08, SD=0.08, n=2, vals=[-0.14 -0.03]

  object:
    Ctrl left: M=0.53, SD=0.39, n=9
    Ctrl right: M=0.64, SD=0.30, n=9
    L-resec (intact RH): M=0.51, SD=0.11, n=3, vals=[0.58 0.57 0.38]
    R-resec (intact LH): M=0.23, SD=0.51, n=2, vals=[ 0.59 -0.14]

  word:
    Ctrl left: M=0.77, SD=0.22, n=8
    Ctrl right: M=-0.09, SD=0.26, n=5
    L-resec (intact RH): M=0.71, SD=0.17, n=2, vals=[0.83 0.59]
    R-resec (intact LH): M=0.38, SD=0.53, n=2, vals=[0.75 0.  ]


In [61]:
# ── RDM Distance by hemisphere ──
# Note: controls were averaged across hemispheres in the original computation.
# OTC are intact hemi only. Check if per-hemi ctrl data is available.
print('='*70)
print('RDM Distance')
print('='*70)

rdm_id = 'subject_id' if 'subject_id' in df_rdm.columns else (
    'subject' if 'subject' in df_rdm.columns else 'sub')

for cat in CATEGORIES:
    print(f'\n  {cat}:')
    # Controls (already hemisphere-averaged in df_rdm)
    ctrl = df_rdm[(df_rdm['group'] == 'control') &
                  (df_rdm['category'] == cat)]['rdm_distance']
    if len(ctrl) > 0:
        print(f'    Ctrl (avg LH+RH): M={ctrl.mean():.2f}, SD={ctrl.std():.2f}, n={len(ctrl)}')
    
    # OTC by surgery side
    otc = df_rdm[(df_rdm['group'] == 'OTC') & (df_rdm['category'] == cat)]
    if 'surgery_side' in df_rdm.columns:
        for ss in ['left', 'right']:
            sub = otc[otc['surgery_side'] == ss]['rdm_distance']
            label_r = 'L-resec (intact RH)' if ss == 'left' else 'R-resec (intact LH)'
            if len(sub) > 0:
                print(f'    {label_r}: M={sub.mean():.2f}, SD={sub.std():.2f}, n={len(sub)}, vals={sub.values.round(2)}')
    else:
        if len(otc) > 0:
            print(f'    OTC all: M={otc["rdm_distance"].mean():.2f}, SD={otc["rdm_distance"].std():.2f}, n={len(otc)}, vals={otc["rdm_distance"].values.round(2)}')
        print('    (No surgery_side column — cannot split L/R resec)')
        print('    To get per-hemi control values, recompute RDM without averaging hemispheres')

RDM Distance

  face:
    Ctrl (avg LH+RH): M=1.14, SD=0.45, n=9
    L-resec (intact RH): M=0.82, SD=0.47, n=3, vals=[1.35 0.43 0.69]
    R-resec (intact LH): M=1.78, SD=0.98, n=2, vals=[1.09 2.48]

  house:
    Ctrl (avg LH+RH): M=1.73, SD=0.83, n=9
    L-resec (intact RH): M=1.59, SD=0.44, n=3, vals=[2.09 1.28 1.4 ]
    R-resec (intact LH): M=2.90, SD=0.61, n=2, vals=[2.47 3.33]

  object:
    Ctrl (avg LH+RH): M=1.20, SD=0.72, n=9
    L-resec (intact RH): M=1.72, SD=0.43, n=3, vals=[1.55 2.21 1.39]
    R-resec (intact LH): M=1.47, SD=0.08, n=2, vals=[1.53 1.41]

  word:
    Ctrl (avg LH+RH): M=1.25, SD=0.60, n=8
    L-resec (intact RH): M=1.28, SD=1.27, n=2, vals=[0.38 2.18]
    R-resec (intact LH): M=1.94, SD=1.54, n=2, vals=[0.85 3.03]


In [73]:
# Sub-ROI breakdown — Geometry Preservation
import pandas as pd, numpy as np

geo = pd.read_csv('/user_data/csimmon2/sym_pt/group_results/geometry/geometry_differential.csv')

SUB_ROIS = ['face_FFA','face_STS','house_PPA','house_TOS',
            'object_LOC','object_pF','word_VWFA','word_STG','evc']
PARENT = ['face','house','object','word']

print('='*90)
print('GEOMETRY PRESERVATION — Sub-ROI Breakdown')
print('='*90)

for cat in PARENT + SUB_ROIS:
    c = geo[geo['category'] == cat]
    if 'geometry_preservation' not in c.columns:
        # Check what the value column is called
        print(f'\nColumns: {c.columns.tolist()}')
        break
    
    print(f'\n  {cat}:')
    for hl in ['left', 'right']:
        ctrl = c[(c['status'] == 'control') & (c['hemi_label'] == hl)]['geometry_preservation']
        if len(ctrl) > 0:
            print(f'    Ctrl {hl}: M={ctrl.mean():.3f}, SD={ctrl.std():.3f}, n={len(ctrl)}')
    
    otc = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact')]
    for ss in ['left', 'right']:
        sub = otc[otc['surgery_side'] == ss]['geometry_preservation']
        label = 'L-resec (intact RH)' if ss == 'left' else 'R-resec (intact LH)'
        if len(sub) > 0:
            print(f'    {label}: M={sub.mean():.3f}, SD={sub.std():.3f}, n={len(sub)}, vals={sub.values.round(3)}')

GEOMETRY PRESERVATION — Sub-ROI Breakdown

  face:
    Ctrl left: M=0.652, SD=0.248, n=9
    Ctrl right: M=0.744, SD=0.305, n=9
    L-resec (intact RH): M=0.516, SD=0.714, n=4, vals=[ 0.787 -0.552  0.911  0.917]
    R-resec (intact LH): M=0.504, SD=0.379, n=2, vals=[0.772 0.236]

  house:
    Ctrl left: M=0.486, SD=0.596, n=9
    Ctrl right: M=0.399, SD=0.528, n=9
    L-resec (intact RH): M=0.256, SD=0.547, n=4, vals=[-0.489  0.454  0.253  0.807]
    R-resec (intact LH): M=-0.085, SD=0.082, n=2, vals=[-0.143 -0.027]

  object:
    Ctrl left: M=0.531, SD=0.395, n=9
    Ctrl right: M=0.637, SD=0.297, n=9
    L-resec (intact RH): M=0.615, SD=0.231, n=4, vals=[0.578 0.933 0.567 0.381]
    R-resec (intact LH): M=0.226, SD=0.514, n=2, vals=[ 0.59  -0.137]

  word:
    Ctrl left: M=0.767, SD=0.215, n=8
    Ctrl right: M=-0.087, SD=0.260, n=5
    L-resec (intact RH): M=0.786, SD=0.182, n=3, vals=[0.83  0.942 0.586]
    R-resec (intact LH): M=0.375, SD=0.528, n=2, vals=[0.749 0.002]

  face_FFA

In [74]:
# Sub-ROI breakdown — RDM Distance (recomputed per hemisphere)
from scipy.spatial.distance import euclidean

pc = pd.read_csv('/user_data/csimmon2/sym_pt/group_results/liu_distinctiveness/pairwise_correlations_differential.csv')
PAIRS = ['face-house','face-object','face-word','house-object','house-word','object-word']

print('='*90)
print('RDM DISTANCE — Sub-ROI Breakdown (per hemisphere)')
print('='*90)

for cat in PARENT + SUB_ROIS:
    cat_df = pc[pc['category'] == cat]
    if len(cat_df) == 0:
        continue
    
    # Get longitudinal subjects per hemi
    sess_counts = cat_df.groupby(['subject_id','hemi_label'])['session'].nunique()
    long_subs = sess_counts[sess_counts >= 2].reset_index()[['subject_id','hemi_label']]
    
    results = []
    for _, row in long_subs.iterrows():
        sid, hl = row['subject_id'], row['hemi_label']
        sub_df = cat_df[(cat_df['subject_id'] == sid) & (cat_df['hemi_label'] == hl)]
        sessions = sorted(sub_df['session'].unique())
        t1, tlast = sessions[0], sessions[-1]
        
        vec_t1, vec_tl = [], []
        for pair in PAIRS:
            v1 = sub_df[(sub_df['session'] == t1) & (sub_df['pair'] == pair)]['fisher_r']
            vl = sub_df[(sub_df['session'] == tlast) & (sub_df['pair'] == pair)]['fisher_r']
            if len(v1) == 1 and len(vl) == 1:
                vec_t1.append(v1.values[0])
                vec_tl.append(vl.values[0])
        
        if len(vec_t1) == 6:
            results.append({
                'subject_id': sid, 'hemi_label': hl,
                'group': sub_df['group'].iloc[0],
                'surgery_side': sub_df['surgery_side'].iloc[0],
                'rdm_distance': euclidean(vec_t1, vec_tl)
            })
    
    if not results:
        continue
    
    rdf = pd.DataFrame(results)
    print(f'\n  {cat}:')
    for hl in ['left', 'right']:
        ctrl = rdf[(rdf['group'] == 'control') & (rdf['hemi_label'] == hl)]['rdm_distance']
        if len(ctrl) > 0:
            print(f'    Ctrl {hl}: M={ctrl.mean():.3f}, SD={ctrl.std():.3f}, n={len(ctrl)}')
    
    otc = rdf[(rdf['group'] == 'OTC') & (rdf['hemi_label'] == 'intact')]
    for ss in ['left', 'right']:
        sub = otc[otc['surgery_side'] == ss]['rdm_distance']
        label = 'L-resec (intact RH)' if ss == 'left' else 'R-resec (intact LH)'
        if len(sub) > 0:
            print(f'    {label}: M={sub.mean():.3f}, SD={sub.std():.3f}, n={len(sub)}, vals={sub.values.round(3)}')

RDM DISTANCE — Sub-ROI Breakdown (per hemisphere)

  face:
    Ctrl left: M=1.127, SD=0.456, n=9
    Ctrl right: M=1.156, SD=0.686, n=9
    L-resec (intact RH): M=1.457, SD=1.324, n=4, vals=[1.351 3.356 0.435 0.686]
    R-resec (intact LH): M=1.783, SD=0.983, n=2, vals=[1.088 2.478]

  house:
    Ctrl left: M=1.517, SD=0.884, n=9
    Ctrl right: M=1.943, SD=1.181, n=9
    L-resec (intact RH): M=1.552, SD=0.364, n=4, vals=[2.089 1.44  1.283 1.398]
    R-resec (intact LH): M=2.904, SD=0.608, n=2, vals=[2.474 3.334]

  object:
    Ctrl left: M=1.362, SD=1.090, n=9
    Ctrl right: M=1.039, SD=0.645, n=9
    L-resec (intact RH): M=1.587, SD=0.437, n=4, vals=[1.547 1.2   2.207 1.393]
    R-resec (intact LH): M=1.467, SD=0.084, n=2, vals=[1.527 1.407]

  word:
    Ctrl left: M=1.089, SD=0.756, n=8
    Ctrl right: M=1.375, SD=0.493, n=5
    L-resec (intact RH): M=1.548, SD=1.011, n=3, vals=[0.382 2.088 2.175]
    R-resec (intact LH): M=1.943, SD=1.539, n=2, vals=[0.855 3.031]

  face_FFA:
    

In [75]:
# Sub-ROI breakdown — Liu Distinctiveness (cross-sectional)
print('='*90)
print('LIU DISTINCTIVENESS — Sub-ROI Breakdown (cross-sectional)')
print('='*90)

liu_path = '/user_data/csimmon2/sym_pt/group_results/liu_distinctiveness/'
import glob
liu_files = glob.glob(f'{liu_path}liu_distinctiveness_*.csv')
print(f'Available: {liu_files}')

# Use differential
liu = pd.read_csv(f'{liu_path}liu_distinctiveness_differential.csv')
print(f'Columns: {liu.columns.tolist()}')
print(f'Categories: {sorted(liu["category"].unique())}')

vcol = [c for c in liu.columns if 'distinct' in c.lower() or 'liu' in c.lower()]
print(f'Value column candidates: {vcol}')

LIU DISTINCTIVENESS — Sub-ROI Breakdown (cross-sectional)
Available: ['/user_data/csimmon2/sym_pt/group_results/liu_distinctiveness/liu_distinctiveness_cat_vs_scramble.csv', '/user_data/csimmon2/sym_pt/group_results/liu_distinctiveness/liu_distinctiveness_differential.csv', '/user_data/csimmon2/sym_pt/group_results/liu_distinctiveness/liu_distinctiveness_hybrid.csv']
Columns: ['subject', 'subject_id', 'group', 'status', 'surgery_side', 'session', 'hemi', 'hemi_label', 'category', 'cat_type', 'roi_status', 'cope_set', 'liu_distinctiveness', 'peak_z', 'n_voxels', 'sphere_voxels']
Categories: ['evc', 'face', 'face_FFA', 'face_STS', 'house', 'house_PPA', 'house_TOS', 'object', 'object_LOC', 'object_pF', 'word', 'word_STG', 'word_VWFA']
Value column candidates: ['liu_distinctiveness']


In [76]:
# Liu Distinctiveness — Sub-ROI Breakdown (cross-sectional)
liu = pd.read_csv('/user_data/csimmon2/sym_pt/group_results/liu_distinctiveness/liu_distinctiveness_differential.csv')

SUB_ROIS = ['face_FFA','face_STS','house_PPA','house_TOS',
            'object_LOC','object_pF','word_VWFA','word_STG','evc']
PARENT = ['face','house','object','word']

print('='*90)
print('LIU DISTINCTIVENESS — Cross-sectional')
print('='*90)

for cat in PARENT + SUB_ROIS:
    c = liu[(liu['category'] == cat)]
    # Take first session per subject per hemi
    c = c.sort_values('session').groupby(['subject_id','hemi_label']).first().reset_index()
    
    if len(c) == 0:
        continue
    
    print(f'\n  {cat}:')
    for hl in ['left', 'right']:
        ctrl = c[(c['status'] == 'control') & (c['hemi_label'] == hl)]['liu_distinctiveness']
        if len(ctrl) > 0:
            print(f'    Ctrl {hl}: M={ctrl.mean():.3f}, SD={ctrl.std():.3f}, n={len(ctrl)}')
    
    otc = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact')]
    for ss in ['left', 'right']:
        sub = otc[otc['surgery_side'] == ss]['liu_distinctiveness']
        label = 'L-resec (intact RH)' if ss == 'left' else 'R-resec (intact LH)'
        if len(sub) > 0:
            print(f'    {label}: M={sub.mean():.3f}, SD={sub.std():.3f}, n={len(sub)}')
    
    nonotc = c[(c['group'] == 'nonOTC') & (c['hemi_label'] == 'intact')]
    if len(nonotc) > 0:
        print(f'    nonOTC intact: M={nonotc["liu_distinctiveness"].mean():.3f}, SD={nonotc["liu_distinctiveness"].std():.3f}, n={len(nonotc)}')

LIU DISTINCTIVENESS — Cross-sectional

  face:
    Ctrl left: M=0.706, SD=0.368, n=21
    Ctrl right: M=0.710, SD=0.389, n=21
    L-resec (intact RH): M=0.984, SD=0.433, n=8
    R-resec (intact LH): M=0.748, SD=0.334, n=7
    nonOTC intact: M=0.659, SD=0.237, n=9

  house:
    Ctrl left: M=0.479, SD=0.805, n=22
    Ctrl right: M=0.107, SD=0.805, n=22
    L-resec (intact RH): M=0.401, SD=0.676, n=8
    R-resec (intact LH): M=0.613, SD=0.664, n=8
    nonOTC intact: M=0.334, SD=0.592, n=9

  object:
    Ctrl left: M=1.028, SD=0.332, n=22
    Ctrl right: M=1.170, SD=0.530, n=22
    L-resec (intact RH): M=1.230, SD=0.503, n=8
    R-resec (intact LH): M=1.087, SD=0.370, n=8
    nonOTC intact: M=1.224, SD=0.320, n=9

  word:
    Ctrl left: M=0.508, SD=0.360, n=21
    Ctrl right: M=0.387, SD=0.482, n=16
    L-resec (intact RH): M=0.901, SD=0.553, n=7
    R-resec (intact LH): M=0.796, SD=0.599, n=7
    nonOTC intact: M=0.791, SD=0.470, n=9

  face_FFA:
    Ctrl left: M=0.727, SD=0.367, n=21
   

In [77]:
# Liu Distinctiveness — Delta (longitudinal)
print('='*90)
print('DELTA LIU DISTINCTIVENESS — Sub-ROI Breakdown')
print('='*90)

for cat in PARENT + SUB_ROIS:
    c = liu[liu['category'] == cat].copy()
    c['session'] = pd.to_numeric(c['session'], errors='coerce')
    
    # Get first and last session per subject per hemi
    idx_first = c.groupby(['subject_id','hemi_label'])['session'].idxmin()
    idx_last  = c.groupby(['subject_id','hemi_label'])['session'].idxmax()
    
    t1 = c.loc[idx_first].set_index(['subject_id','hemi_label'])
    tl = c.loc[idx_last].set_index(['subject_id','hemi_label'])
    
    # Keep only subjects with >1 session
    multi = t1.index[t1['session'] != tl.loc[t1.index, 'session']]
    if len(multi) == 0:
        continue
    
    t1, tl = t1.loc[multi], tl.loc[multi]
    delta = tl['liu_distinctiveness'] - t1['liu_distinctiveness']
    delta = delta.reset_index()
    delta.columns = ['subject_id', 'hemi_label', 'delta_liu']
    
    # Merge metadata
    meta = c[['subject_id','hemi_label','group','status','surgery_side']].drop_duplicates()
    delta = delta.merge(meta, on=['subject_id','hemi_label'], how='left')
    
    print(f'\n  {cat}:')
    for hl in ['left', 'right']:
        ctrl = delta[(delta['status'] == 'control') & (delta['hemi_label'] == hl)]['delta_liu']
        if len(ctrl) > 0:
            print(f'    Ctrl {hl}: M={ctrl.mean():.3f}, SD={ctrl.std():.3f}, n={len(ctrl)}')
    
    otc = delta[(delta['group'] == 'OTC') & (delta['hemi_label'] == 'intact')]
    for ss in ['left', 'right']:
        sub = otc[otc['surgery_side'] == ss]['delta_liu']
        label = 'L-resec (intact RH)' if ss == 'left' else 'R-resec (intact LH)'
        if len(sub) > 0:
            print(f'    {label}: M={sub.mean():.3f}, SD={sub.std():.3f}, n={len(sub)}, vals={sub.values.round(3)}')

DELTA LIU DISTINCTIVENESS — Sub-ROI Breakdown

  face:
    Ctrl left: M=0.176, SD=0.418, n=9
    Ctrl right: M=-0.108, SD=0.411, n=9
    L-resec (intact RH): M=-0.249, SD=0.634, n=4, vals=[ 0.434 -1.096 -0.115 -0.219]
    R-resec (intact LH): M=-0.588, SD=0.266, n=2, vals=[-0.4   -0.776]

  house:
    Ctrl left: M=-0.234, SD=0.643, n=9
    Ctrl right: M=-0.337, SD=0.987, n=9
    L-resec (intact RH): M=0.014, SD=0.286, n=4, vals=[-0.343  0.07  -0.022  0.35 ]
    R-resec (intact LH): M=-1.133, SD=0.105, n=2, vals=[-1.059 -1.208]

  object:
    Ctrl left: M=0.119, SD=0.547, n=9
    Ctrl right: M=0.086, SD=0.528, n=9
    L-resec (intact RH): M=0.074, SD=0.504, n=4, vals=[ 0.593  0.303 -0.582 -0.019]
    R-resec (intact LH): M=-0.148, SD=0.854, n=2, vals=[ 0.455 -0.752]

  word:
    Ctrl left: M=0.209, SD=0.381, n=8
    Ctrl right: M=-0.202, SD=0.413, n=5
    L-resec (intact RH): M=-0.574, SD=0.600, n=3, vals=[ 0.119 -0.903 -0.938]
    R-resec (intact LH): M=-0.559, SD=1.083, n=2, vals=[ 0.

In [84]:
# Cell 1: Compute Lateralization Index from Controls
# ═══════════════════════════════════════════════════════════════
import sys, numpy as np, pandas as pd
from scipy.stats import spearmanr, pearsonr
import glob

sys.path.insert(0, '/home/csimmon2/repos/sym_pt')
import sym_pt_params

CATEGORIES = ['face', 'house', 'object', 'word']

def compute_li(df, sub_col, hemi_col, group_col, val_col, group_val='control',
               hemi_left='left', hemi_right='right'):
    """Compute per-subject LI = (LH - RH) / (|LH| + |RH|) for controls."""
    ctrl = df[df[group_col] == group_val]
    results = []
    for cat in CATEGORIES:
        c = ctrl[ctrl['category'] == cat]
        for sub in c[sub_col].unique():
            lh = c[(c[sub_col] == sub) & (c[hemi_col] == hemi_left)][val_col]
            rh = c[(c[sub_col] == sub) & (c[hemi_col] == hemi_right)][val_col]
            if len(lh) == 1 and len(rh) == 1:
                l, r = lh.values[0], rh.values[0]
                denom = abs(l) + abs(r)
                li = (l - r) / denom if denom > 0 else 0
                results.append({'category': cat, 'subject': sub, 'LI': li,
                               'LH': l, 'RH': r})
    return pd.DataFrame(results)

# ── Load selectivity ──
sel = pd.read_csv('/user_data/csimmon2/sym_pt/group_results/selectivity/selectivity_summary.csv')
sel = sel.sort_values('ses')
sel_cs = sel.groupby(['sub', 'hemi', 'category']).first().reset_index()

li_vol = compute_li(sel_cs, 'sub', 'hemi', 'group', 'volume')
li_sel = compute_li(sel_cs, 'sub', 'hemi', 'group', 'sum_selec_norm')
li_act = compute_li(sel_cs, 'sub', 'hemi', 'group', 'mean_act')

# ── Load geometry ──
geo = pd.read_csv('/user_data/csimmon2/sym_pt/group_results/geometry/geometry_differential.csv')
li_geo = compute_li(geo, 'subject_id', 'hemi_label', 'status', 'geometry_preservation',
                    group_val='control')

# ── Load Liu distinctiveness (first session) ──
liu = pd.read_csv('/user_data/csimmon2/sym_pt/group_results/liu_distinctiveness/liu_distinctiveness_differential.csv')
liu_cs = liu.sort_values('session').groupby(['subject_id', 'hemi_label', 'category']).first().reset_index()
li_liu = compute_li(liu_cs, 'subject_id', 'hemi_label', 'status', 'liu_distinctiveness',
                    group_val='control')

# ── Print per-metric LI ──
print('=' * 80)
print('CONTROL LATERALIZATION INDEX — LI = (LH - RH) / (|LH| + |RH|)')
print('  +1 = fully left-lateralized, -1 = fully right-lateralized')
print('=' * 80)

for name, li_df in [('Volume', li_vol), ('Sum Selectivity', li_sel),
                     ('Mean Activation', li_act), ('Geometry', li_geo),
                     ('Liu Distinctiveness', li_liu)]:
    print(f'\n  {name}:')
    for cat in CATEGORIES:
        vals = li_df[li_df['category'] == cat]['LI']
        if len(vals) > 0:
            print(f'    {cat:<10} LI = {vals.mean():+.3f} (SD={vals.std():.3f}, n={len(vals)})')

CONTROL LATERALIZATION INDEX — LI = (LH - RH) / (|LH| + |RH|)
  +1 = fully left-lateralized, -1 = fully right-lateralized

  Volume:
    face       LI = -0.123 (SD=0.371, n=24)
    house      LI = -0.085 (SD=0.223, n=24)
    object     LI = +0.057 (SD=0.137, n=24)
    word       LI = +0.443 (SD=0.550, n=24)

  Sum Selectivity:
    face       LI = -0.150 (SD=0.415, n=24)
    house      LI = -0.080 (SD=0.260, n=24)
    object     LI = +0.077 (SD=0.148, n=24)
    word       LI = +0.459 (SD=0.572, n=24)

  Mean Activation:
    face       LI = -0.034 (SD=0.101, n=24)
    house      LI = -0.002 (SD=0.060, n=24)
    object     LI = +0.034 (SD=0.038, n=24)
    word       LI = +0.068 (SD=0.098, n=24)

  Geometry:
    face       LI = -0.029 (SD=0.444, n=9)
    house      LI = -0.111 (SD=0.756, n=9)
    object     LI = -0.036 (SD=0.542, n=9)
    word       LI = +0.860 (SD=0.204, n=5)

  Liu Distinctiveness:
    face       LI = +0.020 (SD=0.385, n=21)
    house      LI = +0.222 (SD=0.722, n=22)
  

In [85]:
# Cell 2: Composite LI + Category-Level Correlations
# ═══════════════════════════════════════════════════════════════

# ── Composite LI (average across metrics) ──
print('=' * 80)
print('COMPOSITE LATERALIZATION INDEX')
print('=' * 80)

composite_li = {}
for cat in CATEGORIES:
    lis = []
    for li_df in [li_vol, li_sel, li_act, li_geo, li_liu]:
        vals = li_df[li_df['category'] == cat]['LI']
        if len(vals) > 0:
            lis.append(vals.mean())
    composite_li[cat] = np.mean(lis)
    print(f'  {cat:<10} composite LI = {composite_li[cat]:+.3f}  '
          f'({len(lis)} metrics averaged)')

# ── Control hemisphere asymmetry summary ──
print(f'\n  Lateralization gradient: ', end='')
sorted_cats = sorted(CATEGORIES, key=lambda c: abs(composite_li[c]), reverse=True)
print(' > '.join([f'{c} ({composite_li[c]:+.3f})' for c in sorted_cats]))

# ── OTC outcome means per category ──
# Peak drift
peak_files = glob.glob('/user_data/csimmon2/sym_pt/group_results/**/peak*differential*.csv', recursive=True)
if not peak_files:
    peak_files = glob.glob('/user_data/csimmon2/sym_pt/group_results/**/peak*.csv', recursive=True)
print(f'\nPeak files found: {peak_files}')

peak = pd.read_csv(peak_files[0])
peak['ses_num'] = pd.to_numeric(peak['ses'], errors='coerce')

# Compute drift
grp_cols = ['sub', 'category', 'hemi']
idx_first = peak.groupby(grp_cols)['ses_num'].idxmin()
idx_last = peak.groupby(grp_cols)['ses_num'].idxmax()
t1p = peak.loc[idx_first].set_index(grp_cols)
tlp = peak.loc[idx_last].set_index(grp_cols)
multi_p = t1p.index[t1p['ses_num'] != tlp.loc[t1p.index, 'ses_num']]
t1p, tlp = t1p.loc[multi_p], tlp.loc[multi_p]

drift_df = pd.DataFrame(index=multi_p)
drift_df['drift'] = np.sqrt(
    (tlp['peak_x_mni'] - t1p['peak_x_mni'])**2 +
    (tlp['peak_y_mni'] - t1p['peak_y_mni'])**2 +
    (tlp['peak_z_mni'] - t1p['peak_z_mni'])**2)
drift_df['group'] = t1p['group']
drift_df['intact_hemi'] = t1p['intact_hemi']
drift_df = drift_df.reset_index()

# OTC intact hemi only
otc_drift = drift_df[(drift_df['group'] == 'OTC') & 
                      (drift_df['hemi'] == drift_df['intact_hemi'])]

# OTC geometry — intact hemi
otc_geo = geo[(geo['group'] == 'OTC') & (geo['hemi_label'] == 'intact')]

# RDM distance — recompute quickly
from scipy.spatial.distance import euclidean
pc = pd.read_csv('/user_data/csimmon2/sym_pt/group_results/liu_distinctiveness/pairwise_correlations_differential.csv')
PAIRS = ['face-house', 'face-object', 'face-word', 'house-object', 'house-word', 'object-word']

rdm_rows = []
for cat in CATEGORIES:
    cat_df = pc[(pc['category'] == cat)]
    sess_counts = cat_df.groupby(['subject_id', 'hemi_label'])['session'].nunique()
    long = sess_counts[sess_counts >= 2].reset_index()
    for _, r in long.iterrows():
        sid, hl = r['subject_id'], r['hemi_label']
        sub = cat_df[(cat_df['subject_id'] == sid) & (cat_df['hemi_label'] == hl)]
        sessions = sorted(sub['session'].unique())
        t1, tl = sessions[0], sessions[-1]
        v1, vl = [], []
        for pair in PAIRS:
            a = sub[(sub['session'] == t1) & (sub['pair'] == pair)]['fisher_r']
            b = sub[(sub['session'] == tl) & (sub['pair'] == pair)]['fisher_r']
            if len(a) == 1 and len(b) == 1:
                v1.append(a.values[0])
                vl.append(b.values[0])
        if len(v1) == 6:
            rdm_rows.append({
                'subject_id': sid, 'hemi_label': hl, 'category': cat,
                'group': sub['group'].iloc[0],
                'surgery_side': sub['surgery_side'].iloc[0] if 'surgery_side' in sub.columns else '',
                'rdm_distance': euclidean(v1, vl)
            })

rdm_df = pd.DataFrame(rdm_rows)
otc_rdm = rdm_df[(rdm_df['group'] == 'OTC') & (rdm_df['hemi_label'] == 'intact')]

# ── Category-level table ──
print('\n' + '=' * 80)
print('CATEGORY-LEVEL: LI vs OTC OUTCOMES')
print('=' * 80)

cat_li, cat_drift, cat_geo, cat_rdm = [], [], [], []
for cat in CATEGORIES:
    cat_li.append(composite_li[cat])
    
    d = otc_drift[otc_drift['category'] == cat]['drift']
    cat_drift.append(d.mean() if len(d) > 0 else np.nan)
    
    g = otc_geo[otc_geo['category'] == cat]['geometry_preservation']
    cat_geo.append(g.mean() if len(g) > 0 else np.nan)
    
    r = otc_rdm[otc_rdm['category'] == cat]['rdm_distance']
    cat_rdm.append(r.mean() if len(r) > 0 else np.nan)

cat_li = np.array(cat_li)
cat_drift = np.array(cat_drift)
cat_geo = np.array(cat_geo)
cat_rdm = np.array(cat_rdm)

print(f'\n  {"Category":<10} {"LI":>8} {"Drift(mm)":>10} {"Geometry":>10} {"RDM Dist":>10}')
print(f'  {"-" * 50}')
for i, cat in enumerate(CATEGORIES):
    print(f'  {cat:<10} {cat_li[i]:>+8.3f} {cat_drift[i]:>10.2f} {cat_geo[i]:>10.3f} {cat_rdm[i]:>10.3f}')

# Correlations (n=4, so low power but informative)
print(f'\n  Category-level correlations (n=4):')
for label, vals in [('Drift', cat_drift), ('Geometry', cat_geo), ('RDM Distance', cat_rdm)]:
    mask = ~np.isnan(vals)
    if mask.sum() >= 3:
        rs, ps = spearmanr(cat_li[mask], vals[mask])
        rp, pp = pearsonr(cat_li[mask], vals[mask])
        print(f'    LI vs {label:<12}  Spearman rho={rs:+.3f} (p={ps:.3f})  '
              f'Pearson r={rp:+.3f} (p={pp:.3f})')

COMPOSITE LATERALIZATION INDEX
  face       composite LI = -0.063  (5 metrics averaged)
  house      composite LI = -0.011  (5 metrics averaged)
  object     composite LI = +0.025  (5 metrics averaged)
  word       composite LI = +0.400  (5 metrics averaged)

  Lateralization gradient: word (+0.400) > face (-0.063) > object (+0.025) > house (-0.011)

Peak files found: ['/user_data/csimmon2/sym_pt/group_results/peak_coords/peak_coords.csv']

CATEGORY-LEVEL: LI vs OTC OUTCOMES

  Category         LI  Drift(mm)   Geometry   RDM Dist
  --------------------------------------------------
  face         -0.063       4.42      0.512      1.566
  house        -0.011      11.63      0.143      2.003
  object       +0.025       5.51      0.485      1.547
  word         +0.400       9.89      0.622      1.706

  Category-level correlations (n=4):
    LI vs Drift         Spearman rho=+0.400 (p=0.600)  Pearson r=+0.424 (p=0.576)
    LI vs Geometry      Spearman rho=+0.400 (p=0.600)  Pearson r=+0.552

In [86]:
# Cell 3: Within-Patient LI vs Outcomes
# ═══════════════════════════════════════════════════════════════

print('=' * 80)
print('WITHIN-PATIENT: Control LI vs Patient Outcomes (per category)')
print('=' * 80)

# Build a table: each row = one patient × one category
otc_subs_geo = geo[(geo['group'] == 'OTC') & (geo['hemi_label'] == 'intact')]['subject_id'].unique()

all_rows = []
for sub in otc_subs_geo:
    for cat in CATEGORIES:
        li_val = composite_li[cat]
        
        # Geometry
        g = geo[(geo['subject_id'] == sub) & (geo['hemi_label'] == 'intact') &
                (geo['category'] == cat)]['geometry_preservation']
        geo_val = g.values[0] if len(g) > 0 else np.nan
        
        # RDM distance
        r = rdm_df[(rdm_df['subject_id'] == sub) & (rdm_df['hemi_label'] == 'intact') &
                    (rdm_df['category'] == cat)]['rdm_distance']
        rdm_val = r.values[0] if len(r) > 0 else np.nan
        
        # Drift — need to match subject naming
        # Try both sub formats
        for sub_fmt in [sub, sub.replace('sub-0', 'sub-0')]:
            d = otc_drift[(otc_drift['sub'] == sub_fmt) & (otc_drift['category'] == cat)]['drift']
            if len(d) > 0:
                drift_val = d.values[0]
                break
        else:
            drift_val = np.nan
        
        all_rows.append({
            'subject': sub, 'category': cat, 'control_LI': li_val,
            'geometry': geo_val, 'rdm_distance': rdm_val, 'drift': drift_val
        })

corr_df = pd.DataFrame(all_rows)

# Print full table
print(f'\n  {"Subject":<12} {"Category":<10} {"Ctrl LI":>8} {"Geometry":>10} {"RDM Dist":>10} {"Drift":>10}')
print(f'  {"-" * 62}')
for _, r in corr_df.iterrows():
    drift_str = f'{r["drift"]:.2f}' if not np.isnan(r['drift']) else '—'
    rdm_str = f'{r["rdm_distance"]:.3f}' if not np.isnan(r['rdm_distance']) else '—'
    geo_str = f'{r["geometry"]:.3f}' if not np.isnan(r['geometry']) else '—'
    print(f'  {r["subject"]:<12} {r["category"]:<10} {r["control_LI"]:>+8.3f} '
          f'{geo_str:>10} {rdm_str:>10} {drift_str:>10}')

# ── Overall correlations across all patient×category observations ──
print(f'\n{"=" * 80}')
print('CORRELATIONS')
print('=' * 80)

print(f'\n  Overall (all patients × categories):')
for label, col in [('Drift', 'drift'), ('Geometry', 'geometry'), ('RDM Distance', 'rdm_distance')]:
    mask = ~np.isnan(corr_df[col])
    if mask.sum() >= 4:
        rs, ps = spearmanr(corr_df.loc[mask, 'control_LI'], corr_df.loc[mask, col])
        rp, pp = pearsonr(corr_df.loc[mask, 'control_LI'].values, corr_df.loc[mask, col].values)
        print(f'    LI vs {label:<12}  rho={rs:+.3f} (p={ps:.3f})  r={rp:+.3f} (p={pp:.3f})  n={mask.sum()}')

# ── Per patient ──
print(f'\n  Per patient (LI vs Geometry, n=4 categories each):')
for sub in sorted(corr_df['subject'].unique()):
    s = corr_df[corr_df['subject'] == sub]
    mask = ~np.isnan(s['geometry'])
    if mask.sum() >= 3:
        rs, ps = spearmanr(s.loc[mask, 'control_LI'], s.loc[mask, 'geometry'])
        print(f'    {sub}: rho={rs:+.3f} (p={ps:.3f})')

# ── Excluding sub-008 ──
print(f'\n  Overall excluding sub-008:')
excl = corr_df[~corr_df['subject'].str.contains('008')]
for label, col in [('Drift', 'drift'), ('Geometry', 'geometry'), ('RDM Distance', 'rdm_distance')]:
    mask = ~np.isnan(excl[col])
    if mask.sum() >= 4:
        rs, ps = spearmanr(excl.loc[mask, 'control_LI'], excl.loc[mask, col])
        rp, pp = pearsonr(excl.loc[mask, 'control_LI'].values, excl.loc[mask, col].values)
        print(f'    LI vs {label:<12}  rho={rs:+.3f} (p={ps:.3f})  r={rp:+.3f} (p={pp:.3f})  n={mask.sum()}')

# ── Absolute LI (degree of lateralization regardless of direction) ──
print(f'\n  Using |LI| (absolute lateralization):')
corr_df['abs_LI'] = corr_df['control_LI'].abs()
for label, col in [('Drift', 'drift'), ('Geometry', 'geometry'), ('RDM Distance', 'rdm_distance')]:
    mask = ~np.isnan(corr_df[col])
    if mask.sum() >= 4:
        rs, ps = spearmanr(corr_df.loc[mask, 'abs_LI'], corr_df.loc[mask, col])
        rp, pp = pearsonr(corr_df.loc[mask, 'abs_LI'].values, corr_df.loc[mask, col].values)
        print(f'    |LI| vs {label:<12}  rho={rs:+.3f} (p={ps:.3f})  r={rp:+.3f} (p={pp:.3f})  n={mask.sum()}')

WITHIN-PATIENT: Control LI vs Patient Outcomes (per category)

  Subject      Category    Ctrl LI   Geometry   RDM Dist      Drift
  --------------------------------------------------------------
  sub-004      face         -0.063      0.772      1.088       3.14
  sub-004      house        -0.011     -0.143      2.474      21.19
  sub-004      object       +0.025      0.590      1.527       5.08
  sub-004      word         +0.400      0.749      0.855       8.17
  sub-008      face         -0.063      0.236      2.478      10.77
  sub-008      house        -0.011     -0.027      3.334      16.41
  sub-008      object       +0.025     -0.137      1.407      16.06
  sub-008      word         +0.400      0.002      3.031       8.76
  sub-010      face         -0.063      0.787      1.351       4.11
  sub-010      house        -0.011     -0.489      2.089       8.74
  sub-010      object       +0.025      0.578      1.547       8.31
  sub-010      word         +0.400      0.830      0.382